# Introduction

---

#  Imports

This section corresponds to the import of all the libraries and modules used in the notebook. Some of these libraries may need to be installed in your Python virtual environment if you haven't done so already.

In [ ]:
from branca.colormap import LinearColormap
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import momepy
from shapely import geometry
from shapely.geometry import mapping
from shapely.geometry import Polygon, MultiPolygon, LineString, MultiLineString, MultiPoint, Point, box
from shapely.ops import unary_union
from shapely.ops import split
from shapely.ops import nearest_points
from shapely.strtree import STRtree
from scipy.spatial import cKDTree
from sklearn.neighbors import BallTree
import neatnet
import folium
import json
import branca
import math
from geopy.distance import geodesic
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
import contextily as ctx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from branca.element import MacroElement
from jinja2 import Template
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex
from pyproj import CRS
import re

---

#  Methodology

This section corresponds to all the methodological steps that were taken to measure street functions and classify them into typologies.
It is divided in three main subsections:
1. Data gathering and processing
2. Calculating street functions
3. Classification of streets into typologies

# 1. Data gathering and processing

## 1.1. OpenStreetMap data retrieving

This subsection describes the process of retrieving all the OpenStreetMap (OSM) data that will be used in the subsequent steps of the methodology.

### 1.1.1. Defining the ``study_area`` parameter and retrieving its polygon

The ``study_area`` parameter corresponds to the polygon that will be used as a bounding limit for all the OpenStreetMap data that will be retrieved. This parameter filters the OSM data and only retrieves features that fall within the specified area. The polygon should be defined in a way that it covers the area of interest for the analysis. Before running the tool, to make sure the analysis is being performed for the desired study area, check [Nominatim](https://nominatim.openstreetmap.org/ui/search.html). Enter in the search bar the name of the area you want to analyze and click on "Search". The map will zoom in to the area you searched for. This can be used to check if the polygon corresponds to the study area. If that is the polygon you wish to analyze, then set the text you've entered in [Nominatim](https://nominatim.openstreetmap.org/ui/search.html) as the parameter in the script. The script will then use this parameter to retrieve the OSM data for the specified area.

In [ ]:
study_area = "Município de Lisboa, Portugal"
local_CRS = "epsg:3763"
speed_limits = {
    "motorway": 120,
    "motorway_link": 60,
    "trunk": 100,
    "trunk_link": 60,
    "primary": 50,
    "primary_link": 50,
    "secondary": 50,
    "secondary_link": 50,
    "tertiary": 50,
    "tertiary_link": 50,
    "residential": 50,
    "unclassified": 50,
    "living_street": 20
}

This snippet will use the ``study_area`` parameter to retrieve the polygon that defines the limits of the study area, and it will be used in the results chapter for better communication of the boundaries of the study area. It converts a place name (the study area) into a GeoDataFrame containing the polygon geometry of that place. The Coordinate Reference System (CRS) of the GeoDataFrame is WGS84 (EPSG:4326), so we convert it to EPSG:3763 (a CRS for the portuguese context) for consistency with the rest of the analysis.

In [ ]:
study_area_gdf = ox.geocode_to_gdf(study_area)
study_area_gdf = study_area_gdf.to_crs(local_CRS)

### 1.1.2. Retrieving the exclusion mask of the study area

This subsection retrieves the exclusion mask of the study area. The exclusion mask will be later used to (potentially, as it's not perfect) ensure that, when the road network is converted into a street centerlines network, it keeps its integrity and does not overlap buildings and other features that delimit the urban corridors. As it can be seen below, the exclusion mask is created by many OSM layers and filtered for many tags in order to retain the features that will present the best results in the output network. It is recommended that the user modifies the exclusion mask layers and tags depending on the context of the study area, however, the ones that were selected should work as a starting point for most contexts.

Firstly, it is defined a custom function that will be used for filtering out the features that are not at ground level, as they are not relevant for this analysis. The function checks if the feature has a ``layer`` tag and if it is equal to 0. If the feature does not have a ``layer`` tag, it is considered to be at ground level, like most OSM features are tagged. The function returns ``True`` for features that are at ground level and ``False`` for those that are not.

In [ ]:
def filter_ground_level(gdf):
    # Ensure 'layer' column exists
    if "layer" not in gdf.columns:
        gdf["layer"] = "0"
    layer_num = pd.to_numeric(gdf["layer"], errors="coerce").fillna(0)
    return gdf[layer_num >= 0]

Then, we define a list of OSM layers and tags that will be used to create the exclusion mask. The layers may include buildings, construction sites, schools, sports pitches, cemeteries, parks, hospitals, among others, depending on how these features were mapped in OSM. Some of these features create noise that make it harder for the simplification algorithm to correctly identify what are actual buildings or areas we might want to keep or just smaller street elements like kiosks or fountains. The tags are used to filter out the features that are not relevant for the exclusion mask.

In [ ]:
# Retrieving buildings
buildings = ox.features_from_place(study_area, tags={"building": True})

# Applying filters to the "building" features
if "building" in buildings.columns:
    buildings = buildings[buildings["building"] != "roof"]
    buildings = buildings[buildings["building"] != "container"]
    buildings = buildings[buildings["building"] != "kiosk"]
    buildings = buildings[buildings["building"] != "memorial"]
    buildings = buildings[buildings["building"] != "service"]
    buildings = buildings[buildings["building"] != "guardhouse"]
    buildings = buildings[buildings["building"] != "train_station"]

if "amenity" in buildings.columns:
    buildings = buildings[buildings["amenity"] != "shelter"]
    buildings = buildings[buildings["amenity"] != "fountain"]
    buildings = buildings[buildings["amenity"] != "toilets"]

if "artwork_type" in buildings.columns:
    buildings = buildings[buildings["artwork_type"] != "statue"]

if "historic" in buildings.columns:
    buildings = buildings[buildings["historic"] != "monument"]
    buildings = buildings[buildings["historic"] != "memorial"]

if "memorial" in buildings.columns:
    buildings = buildings[buildings["memorial"] != "statue"]
    buildings = buildings[buildings["memorial"] != "bust"]

if "shop" in buildings.columns:
    buildings = buildings[buildings["shop"] != "kiosk"]

if "bridge:support" in buildings.columns:
    buildings = buildings[buildings["bridge:support"] != "yes"]
    buildings = buildings[buildings["bridge:support"] != "pier"]
    buildings = buildings[buildings["bridge:support"] != "abutment"]
    buildings = buildings[buildings["bridge:support"] != "lift_pier"]
    buildings = buildings[buildings["bridge:support"] != "pivot_pier"]
    buildings = buildings[buildings["bridge:support"] != "pylon"]
    
# Filtering for ground level features
buildings = filter_ground_level(buildings)

In [ ]:
# Retrieving construction areas
construction = ox.features_from_place(study_area, tags={"landuse": "construction"})

# Filtering for ground level features
construction = filter_ground_level(construction)

In [ ]:
# Retrieving schools
schools = ox.features_from_place(study_area, tags={"amenity": "school"})

# Filtering for ground level features
schools = filter_ground_level(schools)

In [ ]:
# Retrieving pitches
pitches = ox.features_from_place(study_area, tags={"leisure": "pitch"})

# Filtering for ground level features
pitches = filter_ground_level(pitches)

In [ ]:
# Retrieving cemeteries
cemeteries = ox.features_from_place(study_area, tags={"landuse": "cemetery"})

# Filtering for ground level features
cemeteries = filter_ground_level(cemeteries)

After that, the exclusion mask is prepared for geospatial analysis by first reprojecting the several layers into a common coordinate reference system (EPSG:3763) to ensure spatial consistency. EPSG:3763 is different from EPSG:4326 (WGS84) because it uses meters as units, which is more suitable for distance calculations and spatial operations. It is also the CRS for Portugal, where our case study takes place. Then, all the layers are dissolved into a single geometry to create a unified ``exclusion_mask``. This step is crucial to ensure that the exclusion mask accurately represents all the features that need to be considered when simplifying the road network.

In [ ]:
buildings = buildings.to_crs(local_CRS)
construction = construction.to_crs(local_CRS)
schools = schools.to_crs(local_CRS)
pitches = pitches.to_crs(local_CRS)
cemeteries = cemeteries.to_crs(local_CRS)

It then extracts and combines the ``geometry`` columns from these layers into a single DataFrame, effectively aggregating all areas that should be protected from simplification. Keeping just the geometry column is important to reduce memory usage and improve performance in subsequent spatial operations.

In [ ]:
exclusion_mask = gpd.GeoDataFrame(
    pd.concat([
       buildings[['geometry']],
       construction[['geometry']],
       schools[['geometry']],
       pitches[['geometry']],
       cemeteries[['geometry']]    
    ], ignore_index = True)
)

Finally, it uses a unary union operation to merge these geometries into one cohesive shape, which serves as the exclusion mask. This mask can be used in further spatial operations to preserve important urban features during processes like street network simplification.

In [ ]:
exclusion_mask = gpd.GeoSeries(unary_union(exclusion_mask.geometry), crs=local_CRS)

### 1.1.3. Retrieving and cleaning the ``highway`` network of the study area

This subsection retrieves the ``highway`` network of the study area. This network is a representation of the road network in OpenStreetMap, and it includes all types of roads, paths, and other transportation routes.

#### 1.1.3.1. Retrieving and converting to GeoDataFrame

In [ ]:
# Filtering for wanted highway types
cf_highway_types = [
    'motorway',
    'motorway_link',
    'trunk',
    'trunk_link',
    'primary',
    'primary_link',
    'secondary',
    'secondary_link',
    'tertiary',
    'tertiary_link',
    'residential',
    'unclassified',
    'living_street',
    'pedestrian'
]

cf = '["highway"~"{}"]'.format('|'.join(cf_highway_types))

# Filtering out area highway types
cf += cf + '["area"!~"yes"]'
cf += cf + '["area:highway"!~"footway"]'
cf += cf + '["area:highway"!~"path"]'
cf += cf + '["area:highway"!~"cycleway"]'
cf += cf + '["area:highway"!~"steps"]'
cf += cf + '["area:highway"!~"pedestrian"]'

# Extend OSMnx useful tags so edges carry directional, PSV, and cycleway info
extra_way_tags = [
    # directional lane counts
    "lanes:forward", "lanes:backward",
    # PSV numeric counts
    "lanes:psv", "lanes:psv:forward", "lanes:psv:backward",
    # PSV per-lane designation strings
    "psv:lanes", "psv:lanes:forward", "psv:lanes:backward",
    "bus:lanes", "bus:lanes:forward", "bus:lanes:backward",  # sometimes used instead of psv
    # cycleway tags
    "cycleway", "cycleway:both", "cycleway:left", "cycleway:right",
    "cycleway:lanes", "cycleway:lanes:forward", "cycleway:lanes:backward"
]

ox.settings.useful_tags_way = sorted(set(list(ox.settings.useful_tags_way) + extra_way_tags))

In [ ]:
network = ox.graph_from_place(
    study_area,
    custom_filter=cf,
    retain_all=False, 
    simplify=False, 
    truncate_by_edge=True
)

network_gdf = ox.graph_to_gdfs(network, nodes=False, edges=True)

network_gdf = network_gdf[network_gdf.geometry.notnull()]

network_gdf = network_gdf[network_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]

network_gdf = network_gdf.to_crs(local_CRS)

In [ ]:
# Custom filter just for isolated cycleways
cwf = '["highway"~"cycleway"]'

# Filtering out area highway types
cwf += cwf + '["area:highway"!~"cycleway"]'

# Extend OSMnx useful tags so edges carry directional, PSV, and cycleway info
extra_cycleway_tags = ["oneway"]

ox.settings.useful_tags_way = sorted(set(list(ox.settings.useful_tags_way) + extra_cycleway_tags))

In [ ]:
cycleway_network = ox.graph_from_place(
    study_area,
    custom_filter=cwf,
    retain_all=True, 
    simplify=False, 
    truncate_by_edge=True
)

cycleway_network_gdf = ox.graph_to_gdfs(cycleway_network, nodes=False, edges=True)

cycleway_network_gdf = cycleway_network_gdf[cycleway_network_gdf.geometry.notnull()]

cycleway_network_gdf = cycleway_network_gdf[cycleway_network_gdf.geometry.type.isin(['LineString', 'MultiLineString'])]

cycleway_network_gdf = cycleway_network_gdf.to_crs(local_CRS)

In [ ]:
def _to_bool_oneway(s):
    s = pd.Series(s)
    return s.map({True: True, False: False, 'yes': True, 'no': False}).fillna(False)

network_gdf['oneway'] = _to_bool_oneway(network_gdf.get('oneway'))
cycleway_network_gdf['oneway'] = _to_bool_oneway(cycleway_network_gdf.get('oneway'))

#### 1.1.3.2. Defining "oneway"

The ``oneway`` column defines if a segment is a one-way road or not. Although the direction of traffic is not relevant for this analysis (the "link" function of a street is not dependent on the direction of traffic), it is important to define the ``oneway`` column in order to be able to calculate the potential capacity of the network (since it will define the number of ``lanes`` that will be assumed, namely in streets of lower hierarchy).

In [ ]:
# Defining roundabouts as "oneway"=True, if they're empty
network_gdf.loc[network_gdf["junction"].isin(["roundabout"]) & network_gdf["oneway"].isnull(), "oneway"] = True

# Defining "motorway", "motorway_link", "trunk", and "trunk_link" as "oneway"=True, if they're empty
network_gdf.loc[network_gdf["highway"].isin(["motorway", "motorway_link", "trunk", "trunk_link"]) & network_gdf["oneway"].isnull(), "oneway"] = True

# Defining remaining highways as "oneway"= False, if they're empty
network_gdf.loc[network_gdf["oneway"].isnull(), "oneway"] = False

#### 1.1.3.3. Defining "lanes"

The ``lanes`` column defines the number of lanes of each segment. The number of lanes is an important parameter for calculating the potential capacity of the network. The first step is to define the ``lanes`` of lower hierarchical segments (``residential``, ``unclassified`` and ``living_street``). We look for empty rows in the ``lanes`` column for these highway levels. We then define ``1`` lane for one-way roads and ``2`` lanes for two-way roads. This is an assumption, as it is a practice with OSM to have ``oneway`` tagged only for roads that are so.

In [ ]:
network_gdf.loc[
    (network_gdf["lanes"].isnull()) & 
    (network_gdf["highway"].isin(["residential", "unclassified", "living_street"])) & 
    (network_gdf["oneway"] == True), 
    "lanes"
] = 1

network_gdf.loc[
    (network_gdf["lanes"].isnull()) & 
    (network_gdf["highway"].isin(["residential", "unclassified", "living_street"])) &
    (network_gdf["oneway"] == False), 
    "lanes"
] = 2

After this, the number of ``lanes`` of the other classes are defined. Luckily, a very substantial part of the network is residential (and that will fall under the previous simplification), and as for the rest of the network, usually the number of ``lanes`` is correctly tagged in OSM. Even so, for this case, the following code sets the number of ``lanes`` base on a weighted average of the length of the segments with that ``highway`` type.

In [ ]:
# Calculate weighted average of "lanes" for each "highway" type
# Only use rows where "lanes" is not null and "length" is available
valid_lanes = network_gdf[network_gdf["lanes"].notnull() & network_gdf["length"].notnull()].copy()
valid_lanes["lanes"] = pd.to_numeric(valid_lanes["lanes"], errors="coerce")

weighted_avg_lanes = (
    valid_lanes.groupby("highway")[["lanes", "length"]]
    .apply(lambda df: np.average(df["lanes"], weights=df["length"]))
    .round()
    .astype(int)
)

# Fill missing "lanes" values using the rounded weighted average for each "highway"
def fill_lanes(row):
    if pd.isnull(row["lanes"]):
        return weighted_avg_lanes.get(row["highway"], np.nan)
    return row["lanes"]

network_gdf["lanes"] = network_gdf.apply(fill_lanes, axis=1)

In [ ]:
def _to_int(x):
    """Robustly extract an integer from messy OSM values.

    Handles: None/NaN, lists/tuples (takes first non-null),
    strings with separators ("|", ";", ",", "/"), ranges like "2-3",
    embedded numbers ("50 km/h"), and plain numeric types.
    Returns np.nan on failure to indicate missingness.
    """
    try:
        if pd.isna(x):
            return np.nan

        # handle lists/tuples: pick first non-null element
        if isinstance(x, (list, tuple)):
            for e in x:
                if not pd.isna(e):
                    x = e
                    break
            else:
                return np.nan

        # strings: try several sane parsing strategies
        if isinstance(x, str):
            s = x.strip()
            if s == "":
                return np.nan
            # normalize separators to pipe
            s = re.sub(r"[;,/]+", "|", s)

            # if pipe-delimited, prefer the first numeric token
            if "|" in s:
                toks = [t.strip() for t in s.split("|") if t.strip() != ""]
                for t in toks:
                    if re.match(r"^[-+]?[0-9]+(\\.[0-9]+)?$", t):
                        return int(float(t))
                # fall through to try parsing first token
                s = toks[0]

            # ranges like 2-3 -> take min(2,3)
            m = re.match(r"^(?P<a>[-+]?[0-9]+(\\.[0-9]+)?)\\s*[-–]\\s*(?P<b>[-+]?[0-9]+(\\.[0-9]+)?)$", s)
            if m:
                a = float(m.group("a"))
                b = float(m.group("b"))
                return int(min(a, b))

            # extract first numeric occurrence (handles "50 km/h", "50mph")
            m = re.search(r"([-+]?[0-9]+(\\.[0-9]+)?)", s)
            if m:
                return int(float(m.group(1)))

            return np.nan

        # numeric types
        return int(float(x))
    except Exception:
        return np.nan


def _count_psv_from_token_string(s):
    """Count PSV/bus-reserved lanes encoded in token strings.

    Accepts numeric inputs, token lists delimited by '|', ';', ',', or '/'.
    Recognizes explicit dedicated markers ("designated", "exclusive") by default.
    Returns an integer count (0 if none or malformed).

    NOTE: This function is conservative on purpose and only counts tokens that
    unambiguously indicate a dedicated PSV lane. If you want to be more permissive
    (e.g., treat 'bus' or 'yes' as dedicated), extend the `psv_explicit` set.
    """
    if pd.isna(s):
        return 0

    # numeric input (already a count)
    if isinstance(s, (int, float)):
        try:
            if np.isnan(s):
                return 0
            return int(float(s))
        except Exception:
            return 0

    if not isinstance(s, str):
        # try to coerce to number
        try:
            return int(float(s))
        except Exception:
            return 0

    ss = s.strip().lower()
    if ss == "":
        return 0

    # split on common separators
    tokens = [t.strip() for t in re.split(r"[|,;/]+", ss) if t.strip() != ""]
    if not tokens:
        return 0

    # Conservative default: only count tokens that explicitly indicate a dedicated PSV lane.
    psv_explicit = {"designated", "exclusive"}

    count = 0
    for t in tokens:
        # numeric token -> add numeric value
        if re.match(r"^[0-9]+$", t):
            count += int(t)
            continue

        # strip common prefixes
        t_clean = re.sub(r'^(psv:|bus:)', '', t)

        # only accept explicit tokens that unambiguously mark a dedicated PSV lane
        if t_clean in psv_explicit:
            count += 1

    return max(0, int(count))


def _safe_min(a, b):
    """Return a safe non-negative integer min(a, b).

    Semantics:
    - If both a and b are missing (NaN/None), returns np.nan to preserve missingness.
    - Missing values are treated as 0 when only one side is missing.
    - Non-numeric inputs are coerced when possible; on failure returns np.nan.
    """
    try:
        a_missing = pd.isna(a)
        b_missing = pd.isna(b)
        if a_missing and b_missing:
            return np.nan

        a_val = 0 if a_missing else int(float(a))
        b_val = 0 if b_missing else int(float(b))
        return int(max(0, min(a_val, b_val)))
    except Exception:
        return np.nan


In [ ]:
# 1) General lane counts
# Ensure total lanes is numeric to avoid string - float errors
lanes_tot = pd.to_numeric(network_gdf.get("lanes"), errors="coerce")

lf = pd.to_numeric(network_gdf.get("lanes:forward"), errors="coerce")
lb = pd.to_numeric(network_gdf.get("lanes:backward"), errors="coerce")

# Start with explicit directional tags if present
lanes_forward_dir = lf.copy()
lanes_backward_dir = lb.copy()

# Where missing, derive from oneway and total lanes
mask_missing_both = lanes_forward_dir.isna() & lanes_backward_dir.isna()
if mask_missing_both.any():
    # oneway -> all lanes in the forward direction
    oneway_mask = mask_missing_both & (network_gdf["oneway"] == True)
    lanes_forward_dir.loc[oneway_mask]  = lanes_tot.loc[oneway_mask]
    lanes_backward_dir.loc[oneway_mask] = 0

    # two-way -> split total lanes
    tw_mask = mask_missing_both & (network_gdf["oneway"] == False)
    tot = lanes_tot.loc[tw_mask].fillna(0).astype(float)
    split_f = np.floor(tot / 2.0).astype(int)
    split_b = (tot - split_f).astype(int)
    lanes_forward_dir.loc[tw_mask]  = split_f
    lanes_backward_dir.loc[tw_mask] = split_b

# Any remaining single-side NaNs: backfill by difference with total
rem_f = lanes_forward_dir.isna() & lanes_tot.notna()
lanes_forward_dir.loc[rem_f] = (lanes_tot.loc[rem_f] - lanes_backward_dir.loc[rem_f].fillna(0)).clip(lower=0)

rem_b = lanes_backward_dir.isna() & lanes_tot.notna()
lanes_backward_dir.loc[rem_b] = (lanes_tot.loc[rem_b] - lanes_forward_dir.loc[rem_b].fillna(0)).clip(lower=0)

# Final integer, nonnegative
lanes_forward_dir  = lanes_forward_dir.fillna(0).astype(int).clip(lower=0)
lanes_backward_dir = lanes_backward_dir.fillna(0).astype(int).clip(lower=0)

In [ ]:
# 2) PSV-reserved lanes per direction
# Numeric counts first
psv_tot = pd.to_numeric(network_gdf.get("lanes:psv"), errors="coerce")
# ensure Series (preserve alignment with network_gdf)
psv_tot = pd.Series(psv_tot, index=network_gdf.index)

psv_f   = pd.to_numeric(network_gdf.get("lanes:psv:forward"), errors="coerce")
psv_f   = pd.Series(psv_f, index=network_gdf.index)

psv_b   = pd.to_numeric(network_gdf.get("lanes:psv:backward"), errors="coerce")
psv_b   = pd.Series(psv_b, index=network_gdf.index)

# Token strings (fallbacks)
tok_any = network_gdf.get("psv:lanes")
tok_any = pd.Series(tok_any, index=network_gdf.index)

tok_f   = network_gdf.get("psv:lanes:forward")
tok_f   = pd.Series(tok_f, index=network_gdf.index)

tok_b   = network_gdf.get("psv:lanes:backward")
tok_b   = pd.Series(tok_b, index=network_gdf.index)

# Some data uses bus:* instead of psv:*
tok_any_bus = network_gdf.get("bus:lanes")
tok_any_bus = pd.Series(tok_any_bus, index=network_gdf.index)

tok_f_bus   = network_gdf.get("bus:lanes:forward")
tok_f_bus   = pd.Series(tok_f_bus, index=network_gdf.index)

tok_b_bus   = network_gdf.get("bus:lanes:backward")
tok_b_bus   = pd.Series(tok_b_bus, index=network_gdf.index)

# Start with explicit directional numeric counts
psv_forward_dir  = psv_f.copy()
psv_backward_dir = psv_b.copy()

# If only total numeric count exists, apportion by directional lane share
mask_tot_only = psv_forward_dir.isna() & psv_backward_dir.isna() & psv_tot.notna()
if mask_tot_only.any():
    tot_psv = psv_tot.loc[mask_tot_only].astype(float)
    lf_share = lanes_forward_dir.loc[mask_tot_only].replace(0, np.nan)
    lb_share = lanes_backward_dir.loc[mask_tot_only].replace(0, np.nan)
    denom = (lf_share + lb_share)
    f_alloc = np.floor(tot_psv * (lf_share / denom)).fillna(0)
    b_alloc = (tot_psv - f_alloc).clip(lower=0)
    psv_forward_dir.loc[mask_tot_only]  = f_alloc
    psv_backward_dir.loc[mask_tot_only] = b_alloc

# If still NaN, parse token strings per direction
mask_need_tokens_f = psv_forward_dir.isna()
if mask_need_tokens_f.any():
    src = tok_f.where(tok_f.notna(), tok_any).where(lambda s: s.notna(), tok_any_bus)
    psv_forward_dir.loc[mask_need_tokens_f] = src.loc[mask_need_tokens_f].map(_count_psv_from_token_string)

mask_need_tokens_b = psv_backward_dir.isna()
if mask_need_tokens_b.any():
    src = tok_b.where(tok_b.notna(), tok_any).where(lambda s: s.notna(), tok_b_bus)
    psv_backward_dir.loc[mask_need_tokens_b] = src.loc[mask_need_tokens_b].map(_count_psv_from_token_string)

# Default zeros where still missing
psv_forward_dir  = psv_forward_dir.fillna(0).astype(int)
psv_backward_dir = psv_backward_dir.fillna(0).astype(int)

# Cap PSV counts by available lanes per direction
psv_forward_dir  = np.minimum(psv_forward_dir,  lanes_forward_dir).astype(int)
psv_backward_dir = np.minimum(psv_backward_dir, lanes_backward_dir).astype(int)

In [ ]:
# 3) Cycleway-designated lanes per direction to subtract from general traffic
# Rules:
# - cycleway:both=lane -> 1 lane each direction on two-way; on oneway -> 1 forward
# - cycleway=lane -> assume both sides on two-way (1 each); on oneway -> 1 forward
# - cycleway:right=lane -> subtract 1 forward; cycleway:left=lane -> subtract 1 backward (two-way assumption)
# - shared_lane is ignored
cyc_both  = (network_gdf.get("cycleway:both")  == "lane")
cyc_main  = (network_gdf.get("cycleway")       == "lane")
cyc_left  = (network_gdf.get("cycleway:left")  == "lane")
cyc_right = (network_gdf.get("cycleway:right") == "lane")

oneway_series = network_gdf["oneway"].fillna(False)

cycle_forward_dir  = pd.Series(0, index=network_gdf.index, dtype=int)
cycle_backward_dir = pd.Series(0, index=network_gdf.index, dtype=int)

# both=lane
mask_tw_both = (~oneway_series) & cyc_both.fillna(False)
cycle_forward_dir.loc[mask_tw_both]  += 1
cycle_backward_dir.loc[mask_tw_both] += 1

mask_one_both = oneway_series & cyc_both.fillna(False)
cycle_forward_dir.loc[mask_one_both] += 1

# cycleway=lane
mask_tw_main = (~oneway_series) & cyc_main.fillna(False)
cycle_forward_dir.loc[mask_tw_main]  += 1
cycle_backward_dir.loc[mask_tw_main] += 1

mask_one_main = oneway_series & cyc_main.fillna(False)
cycle_forward_dir.loc[mask_one_main] += 1

# sides
mask_right = cyc_right.fillna(False)
mask_left  = cyc_left.fillna(False)

# For two-way, right serves forward, left serves backward
mask_tw = (~oneway_series)
cycle_forward_dir.loc[mask_tw & mask_right]  += 1
cycle_backward_dir.loc[mask_tw & mask_left]  += 1

# For oneway, assume any side lane serves the oneway flow -> forward
cycle_forward_dir.loc[oneway_series & (mask_right | mask_left)] += 1

# Cap by available lanes
cycle_forward_dir  = np.minimum(cycle_forward_dir, lanes_forward_dir).astype(int)
cycle_backward_dir = np.minimum(cycle_backward_dir, lanes_backward_dir).astype(int)

# Adding lanes to the "highway"="cycleway", depending on the directionality of the street
# Build masks for cycleway types (oneway values can be True/False or strings; the script earlier normalised to booleans)
mask_cycleway_oneway = (network_gdf["highway"] == "cycleway") & (network_gdf["oneway"] == True)
mask_cycleway_twoway  = (network_gdf["highway"] == "cycleway") & (network_gdf["oneway"] == False)

# Update the local Series that was initialized above to avoid KeyError when the columns are not yet on network_gdf
cycle_forward_dir.loc[mask_cycleway_oneway] += 1
cycle_forward_dir.loc[mask_cycleway_twoway]  += 1
cycle_backward_dir.loc[mask_cycleway_twoway] += 1

In [ ]:
# Initializing "lanes_cycle_forward" and "lanes_cycle_backward" columns in cycleway_network_gdf
if "lanes_cycle_forward" not in cycleway_network_gdf.columns:
    cycleway_network_gdf["lanes_cycle_forward"] = np.nan
if "lanes_cycle_backward" not in cycleway_network_gdf.columns:
    cycleway_network_gdf["lanes_cycle_backward"] = np.nan

# Adding cycle lanes to the cycleway_network_gdf, depending on the directionality of the segment
# If "oneway" is False, add 1 to both directions; if True, add 1 to forward direction only
mask_cycleway_oneway_cw = (cycleway_network_gdf["highway"] == "cycleway") & (cycleway_network_gdf["oneway"] == True)
mask_cycleway_twoway_cw  = (cycleway_network_gdf["highway"] == "cycleway") & (cycleway_network_gdf["oneway"] == False)

cycleway_network_gdf.loc[mask_cycleway_oneway_cw, "lanes_cycle_forward"] = 1
cycleway_network_gdf.loc[mask_cycleway_oneway_cw, "lanes_cycle_backward"] = 0
cycleway_network_gdf.loc[mask_cycleway_twoway_cw, "lanes_cycle_forward"] = 1
cycleway_network_gdf.loc[mask_cycleway_twoway_cw, "lanes_cycle_backward"] = 1

In [ ]:
# 4) Final outputs
network_gdf["lanes_psv_forward"]      = psv_forward_dir.astype(int)
network_gdf["lanes_psv_backward"]     = psv_backward_dir.astype(int)
network_gdf["lanes_general_forward"]  = (lanes_forward_dir  - psv_forward_dir).clip(lower=0).astype(int)
network_gdf["lanes_general_backward"] = (lanes_backward_dir - psv_backward_dir).clip(lower=0).astype(int)
network_gdf["lanes_cycle_forward"]    = cycle_forward_dir.astype(int)
network_gdf["lanes_cycle_backward"]   = cycle_backward_dir.astype(int)

In [ ]:
# Forcing all "pedestrian" highways to have no psv or general lanes
ped_mask = network_gdf["highway"] == "pedestrian"
network_gdf.loc[ped_mask, "lanes_psv_forward"] = 0
network_gdf.loc[ped_mask, "lanes_psv_backward"] = 0
network_gdf.loc[ped_mask, "lanes_general_forward"] = 0
network_gdf.loc[ped_mask, "lanes_general_backward"] = 0

#### 1.1.3.4. Defining "maxspeed"

The methodology assumes that the ``maxspeed`` column is defined in km/h. The first step is to assign some fix values to some types of network elements, based on the ``highway`` tag. These values are based on common speed limits for these types of roads, but they may vary depending on the country or region. The values can be adjusted based on local regulations or specific knowledge of the area being analyzed.

In [ ]:
# Ensure numeric maxspeed for existing data
network_gdf["maxspeed"] = pd.to_numeric(network_gdf["maxspeed"], errors="coerce")

# Ensure length exists (in meters) for weighted averaging
if "length" not in network_gdf.columns:
    network_gdf["length"] = network_gdf.geometry.length.astype(float)

# 1) Keep existing maxspeed as-is (already numeric)
# 2) Fill missing from user-provided dictionary
mask_missing = network_gdf["maxspeed"].isnull()
if mask_missing.any():
    mapped = network_gdf.loc[mask_missing, "highway"].map(speed_limits)
    mapped = pd.to_numeric(mapped, errors="coerce")
    network_gdf.loc[mask_missing, "maxspeed"] = mapped

# 3) Fallback: mode by highway type
# Compute mode of "maxspeed" for each "highway" type
valid_speed = network_gdf[network_gdf["maxspeed"].notnull()].copy()
valid_speed["maxspeed"] = pd.to_numeric(valid_speed["maxspeed"], errors="coerce")

mode_maxspeed = (
    valid_speed.groupby("highway")["maxspeed"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

# Fill missing "maxspeed" values using the mode for each "highway"
def fill_maxspeed(row):
    if pd.isnull(row["maxspeed"]):
        return mode_maxspeed.get(row["highway"], np.nan)
    return row["maxspeed"]

network_gdf["maxspeed"] = network_gdf.apply(fill_maxspeed, axis=1)
# convert to numeric (ints) where possible
network_gdf["maxspeed"] = pd.to_numeric(network_gdf["maxspeed"], errors="coerce").astype("Float64")

### 1.1.4 Calculating ``highway`` network capacity

In [ ]:
# Defining general lane base capacity per "highway" type,between 600 and 1600 p/h/lane
general_lane_base_capacity = {
    "motorway": 1600,
    "motorway_link": 1200,
    "trunk": 1400,
    "trunk_link": 1000,
    "primary": 1000,
    "primary_link": 800,
    "secondary": 800,
    "secondary_link": 600,
    "tertiary": 600,
    "tertiary_link": 600,
    "unclassified": 600,
    "residential": 600,
    "living_street": 200,
    "pedestrian": 0
}

# Defining PSV lane base capacity per "highway" type, between 1000 and 2800 p/h/lane
psv_lane_base_capacity = {
    "motorway": 2800,
    "motorway_link": 2000,
    "trunk": 2400,
    "trunk_link": 1800,
    "primary": 1800,
    "primary_link": 1500,
    "secondary": 1500,
    "secondary_link": 1200,
    "tertiary": 1200,
    "tertiary_link": 1000,
    "unclassified": 1000,
    "residential": 1000,
    "living_street": 333,
    "pedestrian": 0
}

# Defining cycle lane base capacity (1875 p/h/lane)
cycle_lane_base_capacity = {
    "motorway": 1875,
    "motorway_link": 1875,
    "trunk": 1875,
    "trunk_link": 1875,
    "primary": 1875,
    "primary_link": 1875,
    "secondary": 1875,
    "secondary_link": 1875,
    "tertiary": 1875,
    "tertiary_link": 1875,
    "unclassified": 1875,
    "residential": 1875,
    "living_street": 1875,
    "pedestrian": 1875,
    "cycleway": 1875
}

In [ ]:
# Calculating total base capacity per segment
# 1) Capacity of general lanes, per "highway" type, using general_lane_base_capacity dictionary
base_capacity_general = network_gdf["highway"].map(general_lane_base_capacity).fillna(0) * (network_gdf["lanes_general_forward"] + network_gdf["lanes_general_backward"])

# 2) Capacity of PSV lanes, per "highway" type, using psv_lane_base_capacity dictionary
base_capacity_psv = network_gdf["highway"].map(psv_lane_base_capacity).fillna(0) * (network_gdf["lanes_psv_forward"] + network_gdf["lanes_psv_backward"])

# 3) Capacity of cycle lanes, per "highway" type, using cycle_lane_base_capacity dictionary
# compute for the main network
base_capacity_cycle = network_gdf["highway"].map(cycle_lane_base_capacity).fillna(0) * (network_gdf["lanes_cycle_forward"] + network_gdf["lanes_cycle_backward"])
# compute separately for isolated cycleway geometries so indices do not get misaligned
base_capacity_cycleway = cycleway_network_gdf["highway"].map(cycle_lane_base_capacity).fillna(0) * (cycleway_network_gdf["lanes_cycle_forward"] + cycleway_network_gdf["lanes_cycle_backward"])

# Total base capacity per segment (guard against NaNs by filling with 0)
network_gdf["base_capacity"] = (base_capacity_general + base_capacity_psv + base_capacity_cycle).fillna(0)
cycleway_network_gdf["base_capacity"] = base_capacity_cycleway

## 1.2. Deriving a street centerlines network

This subsection corresponds to the creation of a street centerlines network from the ``highway`` network retrieved from OSM in the previous subsection. The street centerlines network is a simplified representation of urban corridors, represented by a line that runs through the center of the street (hence the name) and will be the base on which the street functions will be calculated.

### 1.2.1. Preparing the network for simplification

Since some of the geometries in the road network will be merged to street centerlines, namely in streets that present multiple carriageways, some of the original network information will be lost. With this being said, the next step removes all columns from the GeoDataFrame except for the ``geometry`` column (where the spatial information is kept) and other columns that retain information related to the corridor, such as ``name`` and ``osmid``. This is done to ensure that only the geometries are passed to the ``neatnet.neatify()`` function, which is responsible for deriving the street centerlines.

In [ ]:
_link_map = {
    "motorway_link": "motorway",
    "trunk_link": "trunk",
    "primary_link": "primary",
    "secondary_link": "secondary",
    "tertiary_link": "tertiary"
}

network_gdf["highway_class"] = network_gdf["highway"].map(_link_map).fillna(network_gdf["highway"])

In [ ]:
highway_priority = [
    "motorway",
    "trunk",
    "primary",
    "secondary",
    "tertiary",
    "residential",
    "unclassified",
    "living_street",
    "pedestrian"
]

### 1.2.2. Simplifying the network

The ``neatnet.neatify()`` function is then called to derive the street centerlines from the road network. This function simplifies the road network by removing unnecessary details while preserving the overall structure and connectivity of the streets. The result is a simplified representation of the street network, which will be used for the analysis. This function inputs as parameters the network GeoDataFrame only with the ``geometry``, ``name`` and ``osmid`` columns and projected in EPSG:3763, and the exclusion mask that was set earlier. The output is a GeoDataFrame containing the street centerlines (``street_lines``).

In [ ]:
street_lines = neatnet.neatify(network_gdf, exclusion_mask = exclusion_mask.geometry,)

### 1.2.3. Correcting the ``_status`` == ``new`` segments of the output network (probing methodology)

During the process of running ``neatnet.neatify()``, the tool is not able to keep some of the data from the original network, namely the new segments whose ``_status`` column is set to ``new``. This happens because, as hinted earlier, these segments result from a merge of multiple original segments during the simplification process and do not have a direct correspondence to any segment of that original network. As a result, they do not inherit any attributes from the original network, and their attribute values are null. To fix this, the following code "probes" the original network to find the nearest segment to each of the new segments. It then copies the attributes from the nearest segment in the original network to the new segment in the simplified network. This way, the new segments will have meaningful attribute values that are consistent with the original network.

Before starting the probing methodology, we give both the original carriageways (``network_gdf_essential``) and the new streets (``street_lines``) unique identifiers. These IDs (``orig_id`` and ``street_id``) will help us keep track of matches later when exploding geometries or merging attributes. Resetting indices ensures the IDs are consistent and reproducible.

In [ ]:
def assign_ids(gdf, col_name):
    gdf = gdf.reset_index(drop=True).copy()
    gdf[col_name] = gdf.index.astype(str)
    return gdf

network_gdf = assign_ids(network_gdf, 'orig_id')
street_lines = assign_ids(street_lines, 'street_id')

New segments are extracted from the ``street_lines`` GeoDataFrame in order to apply the procedure to just the segments that require and save computing time.

In [ ]:
street_lines_new = street_lines.loc[street_lines['_status'] == 'new', ['street_id', '_status', 'geometry']].copy()
street_lines_new = gpd.GeoDataFrame(street_lines_new, crs=street_lines.crs)

Points are then interpolated every 10 meters, and at each point a perpendicular “probe” is drawn about 100 meters long (50 meters to each side). These probes cut across the full width of the street, ensuring that even very wide boulevards intersect with the relevant original carriageways. This approach keeps the benefits of splitting (granularity for attribute assignment) while allowing probes to be generated smoothly along the entire street.

In [ ]:
probe_half_length = 50   # 50 meters to each side (total 100-meter probe)
probe_spacing     = 10   # probe every 10 meters along the line

# dissolve to one geometry per street_id
dissolved = street_lines_new.dissolve(by='street_id', as_index=False)

probe_recs = []
for _, row in dissolved.iterrows():
    sid  = row.street_id
    geom = row.geometry
    branches = geom.geoms if geom.geom_type == 'MultiLineString' else [geom]
    for seg in branches:
        seg_len = seg.length
        if seg_len <= probe_spacing:
            continue
        dists = np.arange(probe_spacing, seg_len, probe_spacing)

        # unit perpendicular from endpoints
        p0, p1 = Point(seg.coords[0]), Point(seg.coords[-1])
        dx, dy = p1.x - p0.x, p1.y - p0.y
        ux, uy = -dy, dx
        norm   = (ux**2 + uy**2)**0.5
        if norm == 0:
            continue
        ux, uy = ux/norm, uy/norm

        for d in dists:
            mid  = seg.interpolate(d)
            end1 = Point(mid.x + ux*probe_half_length, mid.y + uy*probe_half_length)
            end2 = Point(mid.x - ux*probe_half_length, mid.y - uy*probe_half_length)
            probe_recs.append({'street_id': sid, 'geometry': LineString([end2, end1])})

probes = gpd.GeoDataFrame(probe_recs, crs=street_lines_new.crs)
probes['probe_seq'] = probes.groupby('street_id').cumcount()
probes['probe_id']  = probes['street_id'] + '_' + probes['probe_seq'].astype(str)
probes = probes.drop(columns='probe_seq')

Once the ``probes`` have been generated, the next step is to determine which original carriageway segments they intersect. This is done efficiently by first using a spatial index to identify candidate lines that fall within the bounding box of each probe, which greatly reduces the number of geometries that need to be checked. Only the candidates that truly intersect the probe are retained. This process assigns to every probe a list of one or more ``orig_id`` values corresponding to the carriageways that it touches. In practice, this means that each probe effectively “samples” the original dataset, recording which existing lines are encountered when extending across the width of the new street. The result is a set of probe geometries, each linked to the original lines it intersects, which provides the foundation for inferring attributes.

In [ ]:
probes = probes.to_crs(network_gdf.crs)
sidx   = network_gdf.sindex

orig_ids_list = []
for g in probes.geometry:
    cand_idx = list(sidx.intersection(g.bounds))
    if not cand_idx:
        orig_ids_list.append([])
        continue
    cand = network_gdf.iloc[cand_idx]
    hits = cand[cand.intersects(g)]
    orig_ids_list.append(hits['orig_id'].tolist())

probes['orig_ids'] = orig_ids_list

Because a single probe can intersect multiple original carriageways, the results from the previous step are stored as lists of identifiers. To analyze these in a structured way, the data is “exploded”, meaning that each probe–carriageway combination is written out as its own row in the dataset. This ensures that every intersected line is represented explicitly rather than hidden inside a list. Once exploded, the attributes from the original dataset, such as the ``name`` of the street and its ``osmid`` identifier, can be joined directly to each probe–carriageway combination. At this point, the dataset expresses exactly which probe touched which original line, along with the relevant attributes, making it possible to analyze the probes individually before moving on to aggregation.

In [ ]:
exploded = (
    probes[['probe_id','street_id','geometry','orig_ids']]
    .explode('orig_ids')
    .dropna(subset=['orig_ids'])
    .rename(columns={'orig_ids':'orig_id'})
)

merged = exploded.merge(
    network_gdf.drop(columns='geometry'),
    on='orig_id',
    how='left'
)

Each probe now has potentially several attribute candidates, since it may have intersected more than one original line. To resolve these, a probe-level aggregation is performed. For the ``name`` attribute, the statistical mode (the most frequently occurring value) is selected. If no single name dominates and there is a tie, the tied values are preserved as a list so that no information is lost. For ``osmid``, which can naturally represent multiple carriageways in parallel, the union of all unique identifiers encountered by the probe is retained. This produces a clean, consolidated record for each probe: one geometry that carries either a single attribute value or a structured collection of possible values. By summarizing at the probe level, the dataset becomes easier to interpret and ready for the next stage of aggregation at the street scale.

In [ ]:
def mode_or_tied(vals):
    s = pd.Series(vals).dropna()
    if s.empty:
        return None
    m = s.mode()
    if len(m) == 1:
        return m.iloc[0]
    return m.unique().tolist()

def most_frequent_with_priority(series: pd.Series, priority_order) -> object:
    s = pd.Series(series).dropna()
    if s.empty:
        return np.nan

    counts = s.value_counts()
    top = counts.max()
    candidates = counts[counts == top].index.tolist()

    # Build a priority index: lower index => higher priority
    priority_index = {v: i for i, v in enumerate(priority_order)} if priority_order is not None else {}

    def tie_key(v):
        return (priority_index.get(v, float("inf")), str(v))

    return min(candidates, key=tie_key)

def agg_per_probe(df):
    name_agg = mode_or_tied(df['name'])
    osmid_agg = pd.unique(df['osmid'].dropna()).tolist()
    highway_agg = most_frequent_with_priority(df['highway_class'], highway_priority)
    base_cap_sum = pd.to_numeric(df.get('base_capacity'), errors='coerce').fillna(0).sum()
    return pd.Series({
        'name': name_agg,
        'osmid': osmid_agg,
        'highway': highway_agg,
        'base_capacity': base_cap_sum,
    })

probe_attrs = merged.groupby('probe_id').apply(agg_per_probe).reset_index()
probes = probes.merge(probe_attrs, on='probe_id', how='left')

Probe-level results are combined by ``street_id`` to assign attributes to each new street in a way that is both lightweight and deterministic. For ``name``, a majority rule is applied across all probes belonging to the street. If a single value clearly occurs most often, that value is assigned. If there is a tie for most frequent, the tie is not forced to a single winner, and instead the list of tied names is retained exactly as the set of “most frequent” candidates. This preserves information about local ambiguity without introducing additional computation or heuristics. For ``osmid``, the union of all identifiers observed across the street’s probes is kept by design, since multiple parallel carriageways may legitimately belong to the same street; no further tie-breaking is needed.

In [ ]:
def _deterministic_sort(vals):
    return sorted(vals, key=lambda v: str(v))

# 1) Explode probe matches and join to the latest attributes from network_gdf
exploded = (
    probes[['probe_id','street_id','orig_ids']]
    .explode('orig_ids')
    .dropna(subset=['orig_ids'])
    .rename(columns={'orig_ids':'orig_id'})
)

keep_cols = [
    'orig_id','name','osmid','highway','base_capacity',
    'lanes_general_forward','lanes_general_backward',
    'lanes_psv_forward','lanes_psv_backward',
    'lanes_cycle_forward','lanes_cycle_backward'
]
attrs_now = network_gdf[keep_cols].copy()

merged = exploded.merge(attrs_now, on='orig_id', how='left')

# 2) Aggregate per street_id using refreshed attributes
records = []
for sid, grp in merged.groupby('street_id'):
    out = {'street_id': sid}

    # name
    s = pd.Series(grp['name']).dropna().explode().dropna()
    if s.empty:
        out['name'] = None
    else:
        counts = s.value_counts()
        winners = _deterministic_sort(list(counts[counts == counts.max()].index))
        out['name'] = winners[0]

    # osmid
    s = pd.Series(grp['osmid']).dropna().explode().dropna()
    out['osmid'] = _deterministic_sort(pd.unique(s).tolist()) if not s.empty else None

    # highway
    s = pd.Series(grp['highway']).dropna().explode().dropna()
    if s.empty:
        out['highway'] = None
    else:
        counts = s.value_counts()
        winners = _deterministic_sort(list(counts[counts == counts.max()].index))
        out['highway'] = winners[0]

    # base_capacity
    s = pd.Series(grp['base_capacity']).dropna().explode().dropna()
    caps = pd.to_numeric(s, errors='coerce').dropna()
    if caps.empty:
        out['base_capacity'] = np.nan
    else:
        mode_vals = caps.mode()
        out['base_capacity'] = float(mode_vals.iloc[0]) if len(mode_vals) == 1 else float(caps.max())

    # Helper to aggregate lanes with total-based fallback
    def agg_dir(col_fwd, col_bwd):
        f = pd.to_numeric(pd.Series(grp[col_fwd]).dropna().explode(), errors='coerce').dropna()
        b = pd.to_numeric(pd.Series(grp[col_bwd]).dropna().explode(), errors='coerce').dropna()
        tot = (
            pd.to_numeric(pd.Series(grp[col_fwd]).dropna().explode(), errors='coerce').fillna(0)
            + pd.to_numeric(pd.Series(grp[col_bwd]).dropna().explode(), errors='coerce').fillna(0)
        )
        f_med = int(round(f.median())) if not f.empty else 0
        b_med = int(round(b.median())) if not b.empty else 0
        tot_med = int(round(tot.median())) if not tot.empty else (f_med + b_med)

        # Fallbacks: if one side is zero but total > the other side, backfill the missing side
        if f_med == 0 and tot_med > b_med:
            f_med = max(tot_med - b_med, 0)
        if b_med == 0 and tot_med > f_med:
            b_med = max(tot_med - f_med, 0)
        return f_med, b_med

    # general lanes
    gf, gb = agg_dir('lanes_general_forward','lanes_general_backward')
    out['lanes_general_forward']  = gf
    out['lanes_general_backward'] = gb

    # PSV lanes
    pf, pb = agg_dir('lanes_psv_forward','lanes_psv_backward')
    out['lanes_psv_forward']  = pf
    out['lanes_psv_backward'] = pb

    # cycle lanes
    cf, cb = agg_dir('lanes_cycle_forward','lanes_cycle_backward')
    out['lanes_cycle_forward']  = cf
    out['lanes_cycle_backward'] = cb

    records.append(out)

street_attrs = pd.DataFrame.from_records(records)


# 3) Merge into street_lines and finalize lane fields
street_lines = street_lines.merge(street_attrs, on='street_id', how='left')

for c in ['name','osmid','highway','base_capacity',
          'lanes_general_forward','lanes_general_backward',
          'lanes_psv_forward','lanes_psv_backward',
          'lanes_cycle_forward','lanes_cycle_backward']:
    cx, cy = f'{c}_x', f'{c}_y'
    if cx in street_lines.columns and cy in street_lines.columns:
        street_lines[c] = street_lines[cx].combine_first(street_lines[cy])
        street_lines.drop(columns=[cx, cy], inplace=True)

for c in ['lanes_general_forward','lanes_general_backward',
          'lanes_psv_forward','lanes_psv_backward',
          'lanes_cycle_forward','lanes_cycle_backward']:
    street_lines[c] = pd.to_numeric(street_lines.get(c), errors='coerce').fillna(0).astype(int)


The final step is to bring the aggregated attributes back into the main ``street_lines`` dataset. Each ``street_id`` is matched with its inferred values for ``name`` and ``osmid``, and these are merged as new columns. When a segment already has valid attributes, those are kept to avoid overwriting existing information. This ensures that the original integrity of the dataset is preserved while the newly created segments are fully integrated into the network. The result is a corrected street dataset where previously ``_status`` == ``new`` segments now carry meaningful names and identifiers, aligned as closely as possible with the original carriageways.

### 1.2.4. Assigning isolated cycle lanes to the street centerlines

Some cycle lanes are mapped in OSM as separate ways alongside the main carriageway, rather than being tagged directly on the road itself. To accurately reflect the presence of cycle infrastructure in the street centerlines network, these separate cycle lane geometries need to be associated with the corresponding street segments. This process involves spatially joining the cycle lane features to the nearest street centerline segments, ensuring that each street segment correctly indicates whether it has adjacent cycle lanes. For that, we do the probe methodology but it a reverse logic, with the cycle lanes being the origin of the probes and the street centerlines being the target of the spatial join.

In [ ]:
# Ensure same CRS
assert street_lines.crs == cycleway_network_gdf.crs, "Project both to same CRS."

# Ensure IDs
if 'street_id' not in street_lines.columns:
    street_lines = street_lines.reset_index(drop=True).assign(street_id=lambda df: df.index.astype(str))
if 'cycle_id' not in cycleway_network_gdf.columns:
    cycleway_network_gdf = cycleway_network_gdf.reset_index(drop=True).assign(cycle_id=lambda df: df.index.astype(str))

# Parameters
probe_half_length   = 20      # meters each side
probe_spacing       = 10      # meters along cycleway
dominance_min_share = 0.5     # require at least 50% of a cycleway’s hits to the chosen street
allow_multi         = False   # allow multiple dominant streets when they tie/meet threshold
weight_by_share     = False   # if True, split capacity by hit-share instead of full credit

# 1) Build probes from CYCLEWAYS
cyc_one = cycleway_network_gdf[['cycle_id','geometry']].dissolve(by='cycle_id', as_index=False)

probe_recs = []
for _, row in cyc_one.iterrows():
    cid  = row.cycle_id
    geom = row.geometry
    parts = geom.geoms if geom.geom_type == 'MultiLineString' else [geom]
    for seg in parts:
        seg_len = seg.length
        if seg_len <= probe_spacing:
            continue
        dists = np.arange(probe_spacing, seg_len, probe_spacing)
        eps = 0.5
        for d in dists:
            t0 = max(d - eps, 0.0)
            t1 = min(d + eps, seg_len)
            p0 = seg.interpolate(t0)
            p1 = seg.interpolate(t1)
            dx, dy = (p1.x - p0.x), (p1.y - p0.y)
            norm = (dx*dx + dy*dy)**0.5
            if norm == 0:
                continue
            ux, uy = -(dy / norm), (dx / norm)  # unit perpendicular
            mid  = seg.interpolate(d)
            end1 = Point(mid.x + ux*probe_half_length, mid.y + uy*probe_half_length)
            end2 = Point(mid.x - ux*probe_half_length, mid.y - uy*probe_half_length)
            probe_recs.append({'cycle_id': cid, 'geometry': LineString([end2, end1])})

probes_cyc = gpd.GeoDataFrame(probe_recs, crs=cycleway_network_gdf.crs)
probes_cyc['seq'] = probes_cyc.groupby('cycle_id').cumcount()
probes_cyc['probe_id'] = probes_cyc['cycle_id'] + '_' + probes_cyc['seq'].astype(str)
probes_cyc = probes_cyc.drop(columns='seq')

# 2) Intersect cycleway probes with STREET network
sidx_st = street_lines.sindex
hit_rows = []
for _, r in probes_cyc.iterrows():
    g = r.geometry
    cand_idx = list(sidx_st.intersection(g.bounds))
    if not cand_idx:
        continue
    cand = street_lines.iloc[cand_idx][['street_id','geometry']]
    hits = cand[cand.intersects(g)]
    for sid in hits['street_id'].tolist():
        hit_rows.append((r.cycle_id, sid))

if not hit_rows:
    # Nothing intersected; create empty columns and stop
    street_lines = street_lines.assign(
        cycle_rev_adj_ids=[[]]*len(street_lines),
        cycle_rev_adj_count=0,
        cycle_rev_base_cap_sum=0.0
    )
else:
    hits_df = pd.DataFrame(hit_rows, columns=['cycle_id','street_id'])

    # 3) For each cycleway, find dominant street(s) by hit share
    counts = (hits_df
              .groupby(['cycle_id','street_id'])
              .size()
              .rename('hits')
              .reset_index())
    totals = counts.groupby('cycle_id')['hits'].sum().rename('total_hits')
    counts = counts.merge(totals, on='cycle_id', how='left')
    counts['share'] = counts['hits'] / counts['total_hits'].replace(0, np.nan)

    # pick dominant candidates
    if allow_multi:
        dom = counts[counts['share'] >= dominance_min_share].copy()
        # if none pass threshold for a cycleway, take the top share street(s)
        no_dom = set(counts['cycle_id']) - set(dom['cycle_id'])
        if no_dom:
            top = (counts[counts['cycle_id'].isin(no_dom)]
                   .sort_values(['cycle_id','share'], ascending=[True, False]))
            top = top.groupby('cycle_id').head(1)
            dom = pd.concat([dom, top], ignore_index=True)
    else:
        dom = (counts.sort_values(['cycle_id','share'], ascending=[True, False])
                     .groupby('cycle_id').head(1))

    # 4) Attribute base_capacity from cycleways to dominant streets
    cap = cycleway_network_gdf[['cycle_id','base_capacity']].copy()
    dom = dom.merge(cap, on='cycle_id', how='left')

    if weight_by_share:
        dom['attrib_capacity'] = pd.to_numeric(dom['base_capacity'], errors='coerce').fillna(0) * dom['share']
    else:
        dom['attrib_capacity'] = pd.to_numeric(dom['base_capacity'], errors='coerce').fillna(0)

    # 5) Aggregate to streets and join
    per_street = dom.groupby('street_id').agg(
        cycle_rev_adj_ids=('cycle_id', lambda s: list(pd.unique(s))),
        cycle_rev_adj_count=('cycle_id', 'nunique'),
        cycle_rev_base_cap_sum=('attrib_capacity', 'sum')
    ).reset_index()

    # Merge new per_street attributes into street_lines
    street_lines = street_lines.merge(per_street, on='street_id', how='left', suffixes=('_old', ''))

    # If older columns exist with '_old' suffix and the new ones weren't created, prefer new but fall back to old.
    if 'cycle_rev_adj_ids' not in street_lines.columns and 'cycle_rev_adj_ids_old' in street_lines.columns:
        street_lines['cycle_rev_adj_ids'] = street_lines['cycle_rev_adj_ids_old']
    if 'cycle_rev_adj_count' not in street_lines.columns and 'cycle_rev_adj_count_old' in street_lines.columns:
        street_lines['cycle_rev_adj_count'] = street_lines['cycle_rev_adj_count_old']
    if 'cycle_rev_base_cap_sum' not in street_lines.columns and 'cycle_rev_base_cap_sum_old' in street_lines.columns:
        street_lines['cycle_rev_base_cap_sum'] = street_lines['cycle_rev_base_cap_sum_old']

    # Ensure consistent types/defaults
    street_lines['cycle_rev_adj_ids'] = street_lines['cycle_rev_adj_ids'].apply(lambda x: x if isinstance(x, list) else [])
    street_lines['cycle_rev_adj_count'] = street_lines['cycle_rev_adj_count'].fillna(0).astype(int)
    street_lines['cycle_rev_base_cap_sum'] = street_lines['cycle_rev_base_cap_sum'].fillna(0.0)

    # === add isolated cycleway lane counts to street_lines ===
    import math
    from shapely.geometry import LineString, MultiLineString

    ALIGN_DIRECTIONS = True  # set False if you only track two-way totals

    # ensure lane cols exist and numeric
    for df, cols in [
        (cycleway_network_gdf, ["lanes_cycle_forward", "lanes_cycle_backward"]),
        (street_lines, ["lanes_cycle_forward", "lanes_cycle_backward"]),
    ]:
        for c in cols:
            if c not in df.columns:
                df[c] = 0
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    # helpers
    def _longest_ls(geom):
        if isinstance(geom, LineString):
            return geom
        if isinstance(geom, MultiLineString):
            return max(geom.geoms, key=lambda g: g.length)
        return None

    def _bearing(ls):
        if not isinstance(ls, LineString) or len(ls.coords) < 2:
            return None
        (x0, y0), (x1, y1) = ls.coords[0], ls.coords[-1]
        return math.atan2(y1 - y0, x1 - x0)

    def _swap_if_opposed(cyc_geom, st_geom, f, b):
        bc = _bearing(_longest_ls(cyc_geom))
        bs = _bearing(_longest_ls(st_geom))
        if bc is None or bs is None:
            return f, b
        return (b, f) if math.cos(bc - bs) < 0 else (f, b)

    # geometry lookups
    street_geom_map = street_lines.set_index("street_id")["geometry"].to_dict()
    cyc_geom_map    = cycleway_network_gdf.set_index("cycle_id")["geometry"].to_dict()

    # bring cycleway lane counts onto dom
    lane_cols = ["lanes_cycle_forward", "lanes_cycle_backward"]
    dom_lanes = dom.merge(cycleway_network_gdf[["cycle_id"] + lane_cols], on="cycle_id", how="left")

    # align per pair, then aggregate to street
    if ALIGN_DIRECTIONS:
        rows = []
        for _, r in dom_lanes.iterrows():
            f = int(r.get("lanes_cycle_forward", 0) or 0)
            b = int(r.get("lanes_cycle_backward", 0) or 0)
            f2, b2 = _swap_if_opposed(cyc_geom_map.get(r["cycle_id"]), street_geom_map.get(r["street_id"]), f, b)
            rows.append((r["street_id"], f2, b2))
        aligned = pd.DataFrame(rows, columns=["street_id","iso_f","iso_b"])
    else:
        aligned = dom_lanes.rename(columns={"lanes_cycle_forward":"iso_f","lanes_cycle_backward":"iso_b"})[["street_id","iso_f","iso_b"]]

    per_street_lanes = (aligned.groupby("street_id", as_index=False)
        .agg(iso_lanes_cycle_forward=("iso_f","sum"),
             iso_lanes_cycle_backward=("iso_b","sum"))
    )

    # merge and add to existing street lane fields
    street_lines = street_lines.merge(per_street_lanes, on="street_id", how="left")
    for src, dst in [("iso_lanes_cycle_forward","lanes_cycle_forward"),
                     ("iso_lanes_cycle_backward","lanes_cycle_backward")]:
        street_lines[src] = pd.to_numeric(street_lines[src], errors="coerce").fillna(0).astype(int)
        street_lines[dst] = pd.to_numeric(street_lines[dst], errors="coerce").fillna(0).astype(int)
        street_lines[dst] = (street_lines[dst] + street_lines[src]).astype(int)

    # cleanup helpers and optional total
    street_lines.drop(columns=["iso_lanes_cycle_forward","iso_lanes_cycle_backward"], inplace=True)
    street_lines["lanes_cycle_total"] = (street_lines["lanes_cycle_forward"] + street_lines["lanes_cycle_backward"]).astype(int)

    # sanity
    assert (street_lines["lanes_cycle_forward"] >= 0).all()
    assert (street_lines["lanes_cycle_backward"] >= 0).all()


# 2. Calculating street functions

## 2.1. Calculating the "link" function

The calculation of the "link" function is very straightforward, corresponding to the edge betweenness centrality (EBC) of each street segment in the network. Betweenness centrality is a measure of how often a node or edge appears on the shortest paths between two other nodes in the network. In the context of street networks, a street segment with high betweenness centrality is one that is frequently used as a route between different locations, indicating its (potential) importance as a through movement corridor, and therefore, with a high "link" function.

### 2.1.1. Creating a NetworkX graph from the street centerlines

There is need for preparing the street centerlines network for the calculation of the betweenness centrality. This involves creating a graph representation of the street network in a format that can be processed by the NetworkX library (a GeoDataFrame won't do). The first step is to define the endpoints of each street segment. The ``endpoints()`` function takes a geometry (either a ``LineString`` or a ``MultiLineString``) and returns the start and end coordinates of these elements. This is important because the graph representation requires nodes (intersections or endpoints) and edges (street segments) to be defined.

In [ ]:
def endpoints(geom):
    if isinstance(geom, LineString):
        c = geom.coords
        return [((c[0][0], c[0][1]), (c[-1][0], c[-1][1]))]
    elif isinstance(geom, MultiLineString):
        pairs = []
        for part in geom.geoms:
            c = part.coords
            pairs.append(((c[0][0], c[0][1]), (c[-1][0], c[-1][1])))
        return pairs
    else:
        return []

After that, the NetworkX graph is created (``street_lines_MultiGraph``). Each street segment is added as an edge in the graph, with its endpoints as nodes. The ``street_id`` and ``length`` attributes are included as edge attributes. 

In [ ]:
street_lines_MultiGraph = nx.Graph()
for sid, geom, L in zip(street_lines["street_id"], street_lines.geometry, street_lines["length"]):
    for a, b in endpoints(geom):   # iterate through all pairs
        street_lines_MultiGraph.add_edge(a, b, street_id=sid, length=float(L))

### 2.1.2. Calculating weighted EBC

Thus, the edge betweenness centrality is calculated using the ``nx.edge_betweenness_centrality()`` function from the NetworkX library. The ``weight`` parameter is set to ``length``, meaning that the shortest paths will be calculated based on the length of the street segments. The ``normalized`` parameter sets if the betweenness values should correspond to the fraction of all-pairs shortest paths that pass through the edge (``True``) or to the number of all-pairs shortest paths that pass through the edge (``False``).

In [ ]:
edge_bc = nx.edge_betweenness_centrality(street_lines_MultiGraph, 
                                         weight = "length",
                                         normalized = "True"
                                         )

### 2.1.3. Mapping EBC values back to centerlines and assigning final "link" results

After that, the edge betweenness centrality that was calculated from the NetworkX graph is mapped back to the original street centerlines GeoDataFrame. For each edge in the graph, it retrieves the ``street_id`` and finds its betweenness centrality score from NetworkX, taking care to handle the fact that undirected edges may be stored as ``(u, v)`` or ``(v, u)``. These values are stored in a dictionary keyed by ``street_id``, which is then used to populate a new ``bet_centrality`` column in the GeoDataFrame. In effect, it ensures every street segment in the dataset carries the centrality measure that was computed on the abstract graph representation.

In [ ]:
id_to_bc = {}

for (u, v, data) in street_lines_MultiGraph.edges(data=True):
    sid = data["street_id"]
    # For an undirected Graph, edge key is (u,v) or (v,u) — use either
    id_to_bc[sid] = edge_bc.get((u, v), edge_bc.get((v, u)))

street_lines["bet_centrality"] = street_lines["street_id"].map(id_to_bc)

Finally, the normalization of the edge betweenness centrality is performed to scale the values between 0 and 1. This is done using 0-max normalization. Unlike min-max normalization, which scales values based on the minimum and maximum values in the dataset, 0-max normalization only considers the maximum value. This means that the lowest value in the dataset will always be 0, and the highest value will be 1, with all other values scaled proportionally in between. This type of normalization was chosen because it preserves the effects the minimum value can have hierarchically. The normalized betweenness centrality values are stored in a new ``LINK`` column, which represents the "link" function of each street segment.

In [ ]:
#maxv = np.nanmax(street_lines["bet_centrality"])
#street_lines["LINK"] = (street_lines["bet_centrality"] / maxv).clip(0, 1)
# Force all null values in link to become "0"
#street_lines["LINK"] = street_lines["LINK"].fillna(0)

## 2.2. Calculating the "place" function

### 2.2.1. Retrieving relevant POI from OSM

The first step in the calculation of the "place" function is to retrieve the relevant points of interest (POI) from OpenStreetMap (OSM) that will be used to calculate the "place" function. POI are features in OSM that represent places where people gather, such as shops, restaurants, schools, parks, and other public facilities. These elements are important for determining the "place" function of streets, as they contribute to the social and economic activity in the street. Most of them are retrieved as points, but some are also retrieved as polygons, depending on the type of feature and how it represents "place" for the streets being analyzed. The cell below lists which POI types are retrieved from OSM and how they are represented (as points or polygons). The user can modify this list depending on the context of the study area and the specific features that are relevant for calculating the "place" function.

In [ ]:
TAGS = {
    "amenity": [
        "bar","biergarten","cafe","fast_food","food_court","pub","restaurant",
        "college","dancing_school","driving_school","first_aid_school","kindergarten",
        "language_school","library","surf_school","toy_library","research_institute",
        "training","music_school","school","traffic_park","university",
        "bicycle_repair_station","bicycle_wash","car_wash","vehicle_inspection","fuel","taxi",
        "bank","bureau_de_change","money_transfer","payment_centre",
        "clinic","dentist","doctors","hospital","nursing_home","pharmacy","social_facility","veterinary",
        "arts_centre","brothel","casino","cinema","community_centre","conference_centre","events_venue",
        "exhibition_centre","fountain","gambling","love_hotel","music_venue","nightclub","planetarium",
        "social_centre","stage","stripclub","studio","swingerclub","theatre",
        "courthouse","fire_station","police","post_office","townhall",
        "animal_shelter","crematorium","dive_centre","funeral_hall","internet_cafe","marketplace",
        "monastery","mortuary","place_of_worship","public_bath"
    ],
    "shop": True,
    "leisure": True,
    "tourism": True,
    "public_transport": [          # <-- move 'platform' here
        "platform"
    ],
    "highway": [
        "bus_stop"                 # <-- leave only bus_stop here
    ],
    "healthcare": True,
    "religion": True
}

AREA_AS_POLYGON = {
    "leisure": {"park","playground","stadium"}
}


,poi_id,element,amenity_norm,shop_norm,public_transport_norm,railway_norm,highway_norm,geometry
0,0,node,NaN,NaN,stop_position,NaN,NaN,POINT (-1023091.788 4682898.673)
1,1,node,NaN,NaN,stop_position,NaN,NaN,POINT (-1023445.16 4682592.937)
2,2,node,NaN,NaN,NaN,switch,NaN,POINT (-1021572.9 4680617.945)
3,3,node,NaN,NaN,NaN,signal,NaN,POINT (-1021584.077 4680683.029)
4,4,node,NaN,NaN,stop_position,stop,NaN,POINT (-1020424.94 4684523.197)


In [ ]:
# ==== 2.2.A″ — Fetch EXACTLY TAGS, normalize, split areas vs points ====

ox.settings.use_cache = True
ox.settings.log_console = False

# Assumes you already have:
# - street_lines (GeoDataFrame in a metric CRS)
# - study_area (string name used elsewhere in your notebook, e.g., "Município de Lisboa, Portugal")

# 1) Pull POIs by portable keys
POI_KEYS = [
    "amenity", "shop", "tourism", "leisure", "office",
    "healthcare", "public_transport", "railway"
]
poi_tags = {k: True for k in POI_KEYS}

pois_any = ox.features_from_place(study_area, tags=poi_tags)
bus_stop_nodes = ox.features_from_place(study_area, tags={"highway": "bus_stop"})

# Optional: filter out sub-surface features if a 'layer' tag exists
def _filter_ground_level(gdf):
    if "layer" not in gdf.columns:
        return gdf
    layer_num = pd.to_numeric(gdf["layer"], errors="coerce").fillna(0)
    return gdf[layer_num >= 0]

pois_any       = _filter_ground_level(pois_any)
bus_stop_nodes = _filter_ground_level(bus_stop_nodes)

# 2) Merge and project to the CRS of street_lines (meters)
pois_all = pd.concat(
    [pois_any, bus_stop_nodes[~bus_stop_nodes.index.isin(pois_any.index)]],
    axis=0
)
pois_all = gpd.GeoDataFrame(pois_all, geometry="geometry", crs="EPSG:4326").to_crs(street_lines.crs)

# 3) Convert areas/lines to representative points so association is uniform
def _to_point(geom):
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, Point):
        return geom
    try:
        return geom.representative_point()
    except Exception:
        return None

pois_all["geometry"] = pois_all["geometry"].apply(_to_point)
pois_all = pois_all[pois_all["geometry"].notna()].copy()

# 4) Normalize common OSM columns (handle list-like values)
def _norm(v):
    return v[0] if isinstance(v, list) and len(v) else v

for col in ["amenity","shop","tourism","leisure","office","healthcare","public_transport","railway","highway"]:
    if col in pois_all.columns:
        pois_all[f"{col}_norm"] = pois_all[col].apply(_norm)
    else:
        pois_all[f"{col}_norm"] = np.nan

# 5) Robust POI ID handling for OSMnx (works for MultiIndex or simple Index)
def _build_poi_id(df: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    df_reset = df.reset_index()

    if {"element", "osmid"}.issubset(df_reset.columns):
        base_id = df_reset["element"].astype(str) + "/" + df_reset["osmid"].astype(str)
    elif {"level_0", "level_1"}.issubset(df_reset.columns):
        base_id = df_reset["level_0"].astype(str) + "/" + df_reset["level_1"].astype(str)
    elif "osmid" in df_reset.columns:
        base_id = df_reset["osmid"].astype(str)
    else:
        # Fallback to positional index if OSM ids aren't available
        base_id = pd.Series(np.arange(len(df_reset)), index=df_reset.index).astype(str)

    df_reset["poi_id"] = base_id
    return gpd.GeoDataFrame(df_reset, geometry="geometry", crs=df.crs)

pois_all = _build_poi_id(pois_all)

# Quick sanity peek
display(pois_all.head()[[
    "poi_id",
    *(["element"] if "element" in pois_all.columns else []),
    *(["osmid"] if "osmid" in pois_all.columns else []),
    "amenity_norm","shop_norm","public_transport_norm","railway_norm","highway_norm","geometry"
]])



The POI are then retrieved from OSM using the ``ox.geometries_from_place()`` function from the OSMnx library. The function takes as parameters the ``study_area`` and the custom filters defined in the previous step. The process is done for both point and polygon features, resulting in two separate GeoDataFrames: ``pois_points`` and ``pois_polygons``.

In [ ]:
# ==== 2.2.B — Variable buffers per segment ====

# Ensure single primary highway value per segment (your aggregation already did this, but be defensive)
def _primary_highway(val):
    if isinstance(val, list) and len(val):
        return val[0]
    return val

street_lines["highway_primary"] = street_lines["highway"].apply(_primary_highway)

pois_polygons = ox.features_from_place(study_area, tags=poi_poly_tags, which_result=None)
# Keep only polygon geometries
pois_polygons = pois_polygons[pois_polygons.geometry.type.isin(['Polygon', 'MultiPolygon'])].copy()

In [ ]:
pois_points = pois_points.to_crs(local_CRS)
pois_polygons = pois_polygons.to_crs(local_CRS)

It is normal that big polygon POI may cover larger areas in which other POI are located (such as a playground and a cafe or kiosk inside a park, for example). In order to capture those elements with a separate "place" value, while also being representative of the "placeness" of the larger polygon, the following code merges the big polygons with the smaller POI that are located within them. Since the methodology will be based on frequency of POI per street length, merging these smaller POI into the larger polygon will allocate to the latter the "place" value of the former.

In [ ]:
def merge_pois_into_polygons(pois_points, pois_polygons):
    # Ensure CRS match
    if pois_points.crs != pois_polygons.crs:
        pois_points = pois_points.to_crs(pois_polygons.crs)

    records, visited = [], set()

    for idx, seed in pois_polygons.iterrows():
        if idx in visited:
            continue

        # --- find connected component (closure of intersections) ---
        comp = pois_polygons[pois_polygons.geometry.intersects(seed.geometry)]
        prev_len = -1
        while len(comp) != prev_len:
            prev_len = len(comp)
            comp_union = unary_union(comp.geometry)
            comp = pois_polygons[pois_polygons.geometry.intersects(comp_union)]

        visited.update(comp.index)

        # --- dissolve geometry (everything contributes to shape) ---
        dissolved = unary_union(comp.geometry)

        # --- identify parks and filter gardens that overlap them ---
        leisure = comp.get("leisure", "").astype(str).str.lower()
        has_park = leisure.eq("park").any()
        park_union = unary_union(comp.loc[leisure.eq("park"), "geometry"]) if has_park else None

        filtered_rows = []
        for i, pg in comp.iterrows():
            ltype = str(pg.get("leisure", "")).lower()
            # exclude gardens that overlap parks by positive area
            if ltype == "garden" and park_union is not None:
                inter = pg.geometry.intersection(park_union)
                if getattr(inter, "area", 0.0) > 0.0:
                    continue
            filtered_rows.append(pg)

        # --- deduplicate by leisure type (keep only one polygon per type) ---
        best = {}
        for _, pg in gpd.GeoDataFrame(filtered_rows).iterrows():
            ltype = str(pg.get("leisure", "")).lower() or "_unknown"
            area = getattr(pg.geometry, "area", 0.0)
            cur = best.get(ltype)
            if (cur is None) or (area > getattr(cur.geometry, "area", 0.0)):
                best[ltype] = pg
        dedup_rows = list(best.values())

        # --- points covered by dissolved geometry (boundary-inclusive) ---
        pts_in = pois_points[pois_points.geometry.apply(lambda g: dissolved.covers(g))]

        # --- collect contributors ---
        contributing = []
        for _, pt in pts_in.iterrows():
            contributing.append({
                "type": "point",
                "geometry": pt.geometry,
                "attributes": pt.drop("geometry").to_dict()
            })
        for pg in dedup_rows:
            contributing.append({
                "type": "polygon",
                "geometry": pg.geometry,
                "attributes": pg.drop("geometry").to_dict()
            })

        records.append({"geometry": dissolved, "contributing_pois": contributing})

    return gpd.GeoDataFrame(records, geometry="geometry", crs=pois_polygons.crs)


merged_pois_polygons = merge_pois_into_polygons(pois_points, pois_polygons)


### 2.2.2. Creating variable buffers for street segments and assigning POI

Next, buffers are created for each street segment in order to assign the POI to each street. The buffers are created with a variable distance depending on the type of street segment, determined by the ``highway`` tag in OSM. Different types of streets have different widths and characteristics, so using a variable buffer distance allows for a more accurate representation of the area that each street segment influences. For example, a motorway will have a larger buffer distance compared to a residential street, reflecting its wider physical presence and greater influence on surrounding POI. This approach helps to ensure that the assignment of POI to street segments is contextually appropriate and reflects the real-world spatial relationships between streets and points of interest. Besides that, the reach of the buffers is also boosted 1.15 times for segments whose ``status`` is set to ``new``, since these segments may represent larger streets that were simplified from multiple carriageways, and therefore, may require a larger buffer to capture all relevant POI.

In [ ]:
# Define buffer sizes for each highway type
buffer_sizes = {
    "motorway": 60,
    "trunk": 55,
    "primary": 50,
    "secondary": 45,
    "tertiary": 40,
    "residential": 35,
    "unclassified": 35,
    "living_street": 30,
    "pedestrian": 30
}

def _buffer_radius(row):
    base = buffer_by_highway.get(row["highway_primary"], 35.0)
    # bonus for _status == "new"
    return base * (1.15 if row.get("_status") == "new" else 1.0)

street_lines["place_buffer_m"] = street_lines.apply(_buffer_radius, axis=1)

local_EPSG = CRS.from_user_input(local_CRS).to_epsg()

if street_lines.crs is None or street_lines.crs.to_epsg() != local_EPSG:
    street_lines_buffered = street_lines.to_crs(local_CRS).copy()
else:
    street_lines_buffered = street_lines.copy()

# Trigger spatial index (speed-up)
_ = place_buffers.sindex


The next cell projects the point and polygon POI layers to the CRS used for the street buffers, flags POI that are permitted to count on high-speed corridors (motorway/trunk) using a small whitelist (fuel, bus_stop, platform), assigns a stable poi_id (stringified index), and concatenates the two POI tables into a single GeoDataFrame with geometry, poi_id and is_allowed columns; it then performs a spatial join to find which POI intersect each street buffer, removes POI that are not permitted on motorway/trunk segments, aggregates the remaining poi_id values by street_id into lists, and finally writes that result back to the main street_lines table as the new poi_ids column (empty list for segments with no matching POI) so subsequent steps can compute POI presence

In [ ]:
# ==== 2.2.C — Associate POIs to segments with eligibility rules (no entropy artifacts) ====

# 1) Rough spatial join: POI inside a segment’s buffer
poi_in_buffers = gpd.sjoin(
    pois_all[["poi_id","geometry","amenity_norm","shop_norm","tourism_norm","leisure_norm",
              "office_norm","healthcare_norm","public_transport_norm","railway_norm","highway_norm"]],
    place_buffers[["street_id","highway_primary","_status","place_buffer_m","geometry"]],
    predicate="within",
    how="inner"
).reset_index(drop=True)

# 2) If a POI hits multiple buffers, attach it to the nearest centerline
_seg_geom = street_lines.set_index("street_id")["geometry"].to_dict()

def _dist_to_seg(row):
    geom = _seg_geom.get(row["street_id"])
    return row.geometry.distance(geom) if geom is not None else np.inf

poi_in_buffers["dist_to_seg"] = poi_in_buffers.apply(_dist_to_seg, axis=1)
poi_nearest = (
    poi_in_buffers.sort_values(["poi_id","dist_to_seg"])
                  .groupby("poi_id", as_index=False)
                  .first()
)

# 3) Enforce motorway/trunk rule:
#    - On motorway/trunk: only amenity=fuel and bus stops
#    - Else: accept everything we collected

def _is_bus_stop_like(r):
    if r.get("highway_norm") == "bus_stop":
        return True
    return r.get("public_transport_norm") in {"platform","stop_position"}

def _poi_allowed(hwy_primary, r):
    if hwy_primary in {"motorway","trunk"}:
        if r.get("amenity_norm") == "fuel": return True
        if _is_bus_stop_like(r): return True
        return False
    return True

mask_allowed = poi_nearest.apply(lambda r: _poi_allowed(r["highway_primary"], r), axis=1)
place_poi_attached = poi_nearest.loc[mask_allowed].copy()

# 4) Build a per-street table of POIs (counts only — no groups/entropy)
place_segment_pois = (
    place_poi_attached
    .groupby("street_id")
    .agg(
        place_poi_ids=("poi_id", list),
        place_poi_count=("poi_id", "count")
    )
    .reset_index()
)

# Assign to street_lines as a new column
street_lines["poi_ids"] = street_lines["street_id"].map(poi_ids_by_street).apply(lambda x: x if isinstance(x, list) else [])

,street_id,highway_primary,_status,place_poi_count
998,998,primary,changed,105
4767,4767,residential,original,83
5428,5428,residential,original,75
9863,9863,secondary,new,73
1443,1443,residential,changed,72
175,175,residential,changed,71
3959,3959,tertiary,original,66
1410,1410,residential,changed,63
123,123,pedestrian,changed,60
373,373,primary,changed,60


### 2.2.3. Adjusting for street type, handling outliers and calculating final "place" results

Then, a set of adjustment factors are defined to account for the influence that different types of streets have on the experience of POI activities from the perspective of the place function. These factors modify the count per 100 meters that is performed on the ``street_lines`` GeoDataFrame. Meaning that streets with, for example, a ``highway`` tag set to ``motorway`` will have their POI count per 100 meters multiplied by 0.5, while streets with a ``highway`` tag set to ``pedestrian`` will have their POI count per 100 meters multiplied by 1.5. The measuring of place in this work follows the GIS-based logic, and this step is a way to reintroduce in the analysis the experience of the street user, who will likely perceive streets differently depending on their type and characteristics.

In [ ]:
# Count POIs per street segment and normalize per 100 meters, with adjustment factors

# 2) Highway adjustment factor (keeps pedestrian/living_street impactful, motorway/trunk damped)
def _highway_factor(h):
    if h == "pedestrian":
        return 1.20
    elif h == "living_street":
        return 1.10
    elif h in {"primary","secondary","tertiary","residential","unclassified"}:
        return 1.0
    elif h in {"motorway","trunk"}:
        return 0.5
    else:
        return 1.0

street_lines["highway_adj"] = street_lines["highway_primary"].apply(_highway_factor)

street_lines["pois_per_100m"] = street_lines.apply(count_pois_per_100m, axis=1)

,street_id,highway_primary,n_pois,highway_adj,PLACE_raw,PLACE
0,0,tertiary,9,1.0,9.0,0.911561
1,1,NaN,1,1.0,1.0,0.500997
2,2,residential,0,1.0,0.0,0.214496
3,3,trunk,0,0.5,0.0,0.214496
4,4,NaN,0,1.0,0.0,0.214496


To address outliers, the following code identifies segments whose POI per 100 meters corresponds to the 99th percentile or higher and performs a winsorization, allocating to them the value of 1 in the place function.

In [ ]:
pct99 = np.nanpercentile(street_lines["pois_per_100m"].astype(float), 99)
max_below = street_lines.loc[street_lines["pois_per_100m"].astype(float) < pct99, "pois_per_100m"].astype(float).max()
if not pd.isna(max_below):
    street_lines.loc[street_lines["pois_per_100m"].astype(float) >= pct99, "pois_per_100m"] = max_below

Finally, the "place" function is calculated by normalizing the adjusted POI density values between 0 and 1 (taking into account the winsorization of the outliers) using 0-max normalization. The normalized values are stored in a new ``PLACE`` column in the ``street_lines`` GeoDataFrame, representing the "place" function of each street segment.

In [ ]:
# 0-max normalize pois_per_100m into PLACE (range 0..1)
max_pois = np.nanmax(street_lines["pois_per_100m"].astype(float))
street_lines["PLACE"] = (street_lines["pois_per_100m"].astype(float) / max_pois).clip(0, 1)
street_lines["PLACE"] = street_lines["PLACE"].fillna(0.0)

# 3. Classification of streets into typologies

As proposed by Jones et al. (2008), streets can be classified into different typologies based on their "link" and "place" functions. This classification helps to understand the role that each street plays in the urban fabric, whether it is primarily a through movement corridor (high "link") or a destination for social and economic activities (high "place"). Part of the value of classifying streets this way lies in the ability to account for both functions in a non-mutually exclusive way. This way, streets are allocated to a 5x5 matrix, where each axis corresponds to one of the functions, and each street segment is classified based on its "link" and "place" values. However, for the analysis of the results, most streets will fall into just a few typologies, as many streets will have either a high "link" function and a low "place" function (e.g., motorways) or a high "place" function and a low "link" function (e.g., pedestrian streets). To take this into account, two ways of setting these typologies are proposed: (1) an absolute typology where the thresholds for the categorization are fixed (0.2, 0.4, 0.6, 0.8); and (2) a relative typology where the thresholds are set based on the distribution of the "link" and "place" values in the dataset (using quintiles). The absolute typology allows for assessing which streets have an indisputable tendency to be either "link" or "place" streets, while the relative typology allows for a more nuanced understanding of the street functions in the context of the specific study area.

In [ ]:
# --- Absolute classification helper (left-closed, right-open; exact edges go to the upper bin) ---
_eps = np.nextafter(1.0, np.inf)  # slightly above 1.0 so 1.0 is captured in the last bin

abs_bins = [0.0, 0.2, 0.4, 0.6, 0.8, _eps]  # [0,0.2) [0.2,0.4) ... [0.8,1.0]
link_labels = pd.CategoricalDtype(categories=["V", "IV", "III", "II", "I"], ordered=True)
place_labels = pd.CategoricalDtype(categories=["E", "D", "C", "B", "A"], ordered=True)

# Classify LINK and PLACE separately
street_lines["LINK_abs"] = pd.cut(
    street_lines["LINK"],
    bins=abs_bins,
    right=False,                # left-closed, right-open -> boundaries go to the upper bin
    labels=link_labels.categories
).astype(link_labels)

street_lines["PLACE_abs"] = pd.cut(
    street_lines["PLACE"],
    bins=abs_bins,
    right=False,
    labels=place_labels.categories
).astype(place_labels)

# Combine into LP_class_abs (e.g., "IV-C")
street_lines["LP_class_abs"] = street_lines["LINK_abs"].astype(str) + "-" + street_lines["PLACE_abs"].astype(str)

In [ ]:
# --- Relative classification using percentile ranks with ties -> upper class ---
# Percentile rank in (0, 1], using method='max' so ties receive the highest rank within the tie group
link_prank = street_lines["LINK"].rank(pct=True, method="max")
place_prank = street_lines["PLACE"].rank(pct=True, method="max")

_eps = np.nextafter(1.0, np.inf)  # ensure rank==1.0 lands in last bin
q_bins = [0.0, 0.2, 0.4, 0.6, 0.8, _eps]  # quintile cutpoints; boundaries go to the upper bin

link_labels = pd.CategoricalDtype(categories=["V", "IV", "III", "II", "I"], ordered=True)
place_labels = pd.CategoricalDtype(categories=["E", "D", "C", "B", "A"], ordered=True)

street_lines["LINK_rel"] = pd.cut(
    link_prank,
    bins=q_bins,
    right=False,  # boundary values (including ties that land exactly on a cut) go to the upper bin
    labels=link_labels.categories
).astype(link_labels)

street_lines["PLACE_rel"] = pd.cut(
    place_prank,
    bins=q_bins,
    right=False,
    labels=place_labels.categories
).astype(place_labels)

# Combine into LP_class_rel (e.g., "IV-C")
street_lines["LP_class_rel"] = street_lines["LINK_rel"].astype(str) + "-" + street_lines["PLACE_rel"].astype(str)


---

#  Results

This section corresponds to the various results that were obtained from the multiple methodologies that were taken in the previous section.

# 4. Base street centerlines network

 The base street centerlines network is the output of the methodological steps taken in section 1.2. It is a simplified representation of urban corridors, represented by a line that runs through the center of the street (hence the name) and will be the base on which the street functions will be calculated. The street centerlines network is a GeoDataFrame that contains, for each segments, OSM attributes such as ``name`` and ``osmid``, as well as a unique identifier for each segment (``street_id``) and a status column (``_status``) that indicates whether the segment was retained from the original network or created as a new segment during the simplification process.

In [ ]:
import folium
from branca.colormap import LinearColormap

# Prepare GeoDataFrame in WGS84 for folium
streets_wgs = street_lines.to_crs(epsg=4326).copy()

# --- Make all non-geometry properties JSON-serializable (convert numpy/pandas scalars to native Python types) ---
def _to_json_safe_gdf(gdf):
    g = gdf.copy()
    # helper to convert a scalar to native python / JSON-safe value
    def _py(x):
        try:
            if pd.isna(x):
                return None
        except Exception:
            pass
        # numpy / pandas scalar types -> native Python
        try:
            import numpy as _np
            if isinstance(x, (_np.integer,)):
                return int(x)
            if isinstance(x, (_np.floating,)):
                return float(x)
            if isinstance(x, (_np.bool_,)):
                return bool(x)
        except Exception:
            pass
        # pandas Timestamp -> ISO string
        try:
            if isinstance(x, pd.Timestamp):
                return x.isoformat()
        except Exception:
            pass
        return x

    for col in g.columns:
        if col == g.geometry.name:
            continue
        g[col] = g[col].apply(_py)
    return g

streets_wgs = _to_json_safe_gdf(streets_wgs)

# Ensure base_capacity numeric for styling
base_cap_series = pd.to_numeric(streets_wgs["base_capacity"], errors="coerce").fillna(0.0)
vmin, vmax = float(base_cap_series.min()), float(base_cap_series.max())

# Build a color scale (adjust palette as desired)
palette = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]
colormap = LinearColormap(palette, vmin=vmin, vmax=vmax)
colormap.caption = "base_capacity (p/h)"

# Center the map on the study area (fallback to streets centroid)
if "study_area_gdf" in globals():
    center_geom = study_area_gdf.to_crs(epsg=4326).geometry.centroid.iloc[0]
else:
    center_geom = streets_wgs.geometry.centroid.unary_union.centroid
m = folium.Map(location=[center_geom.y, center_geom.x], zoom_start=12, tiles="CartoDB positron")

# Tooltip: keep only columns that exist
tooltip = folium.GeoJsonTooltip(
    fields=["name", "highway", "base_capacity","lanes_general_forward","lanes_general_backward",
            "lanes_psv_forward","lanes_psv_backward","lanes_cycle_forward","lanes_cycle_backward"],
    aliases=["Street Name:", "Highway:", "Base Capacity:", "Lanes (General, Forward):", "Lanes (General, Backward):",
             "Lanes (PSV, Forward):", "Lanes (PSV, Backward):", "Lanes (Cycle, Forward):", "Lanes (Cycle, Backward):"],
    localize=True,
    labels=True,
    sticky=True
)

def style_function(feature):
    try:
        val = feature["properties"].get("base_capacity", 0)
        val = float(val) if val is not None else 0.0
    except Exception:
        val = 0.0
    return {
        "color": colormap(val),
        "weight": 3 if val > 0 else 1,
        "opacity": 0.9,
    }

folium.GeoJson(
    data=streets_wgs,
    name="Street centerlines (base_capacity)",
    style_function=style_function,
    tooltip=tooltip,
    highlight_function=lambda feat: {"weight": 5, "color": "#000000"}
).add_to(m)

# Add color scale and layer control
colormap.add_to(m)
folium.LayerControl().add_to(m)

# Save map to HTML file
m.save("street_network_base_capacity_map.html")

In [ ]:
# Use network_gdf as the base instead of street_lines
streets_wgs = network_gdf.to_crs(epsg=4326).copy()

# --- Make all non-geometry properties JSON-serializable (convert numpy/pandas scalars to native Python types) ---
def _to_json_safe_gdf(gdf):
    g = gdf.copy()
    def _py(x):
        try:
            if pd.isna(x):
                return None
        except Exception:
            pass
        try:
            import numpy as _np
            if isinstance(x, (_np.integer,)):
                return int(x)
            if isinstance(x, (_np.floating,)):
                return float(x)
            if isinstance(x, (_np.bool_,)):
                return bool(x)
        except Exception:
            pass
        try:
            if isinstance(x, pd.Timestamp):
                return x.isoformat()
        except Exception:
            pass
        return x

    for col in g.columns:
        if col == g.geometry.name:
            continue
        g[col] = g[col].apply(_py)
    return g

streets_wgs = _to_json_safe_gdf(streets_wgs)

# Ensure base_capacity numeric for styling (fallback to zeros if column missing)
base_cap_series = pd.to_numeric(streets_wgs.get("base_capacity", pd.Series(0.0, index=streets_wgs.index)), errors="coerce").fillna(0.0)
vmin, vmax = float(base_cap_series.min()), float(base_cap_series.max())

# Build a color scale (adjust palette as desired)
palette = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]
colormap = LinearColormap(palette, vmin=vmin, vmax=vmax)
colormap.caption = "base_capacity (p/h)"

# Center the map on the study area (fallback to streets centroid)
if "study_area_gdf" in globals():
    center_geom = study_area_gdf.to_crs(epsg=4326).geometry.centroid.iloc[0]
else:
    center_geom = streets_wgs.geometry.centroid.unary_union.centroid
m = folium.Map(location=[center_geom.y, center_geom.x], zoom_start=12, tiles="CartoDB positron")

# Tooltip: keep only columns that exist (and keep aliases aligned)
desired_fields = ["name", "highway", "base_capacity","lanes_general_forward","lanes_general_backward",
                  "lanes_psv_forward","lanes_psv_backward","lanes_cycle_forward","lanes_cycle_backward"]
desired_aliases = ["Street Name:", "Highway:", "Base Capacity:", "Lanes (General, Forward):", "Lanes (General, Backward):",
                   "Lanes (PSV, Forward):", "Lanes (PSV, Backward):", "Lanes (Cycle, Forward):", "Lanes (Cycle, Backward):"]
fields = [f for f in desired_fields if f in streets_wgs.columns]
aliases = [a for f, a in zip(desired_fields, desired_aliases) if f in streets_wgs.columns]

tooltip = folium.GeoJsonTooltip(
    fields=fields,
    aliases=aliases,
    localize=True,
    labels=True,
    sticky=True
)

def style_function(feature):
    try:
        val = feature["properties"].get("base_capacity", 0)
        val = float(val) if val is not None else 0.0
    except Exception:
        val = 0.0
    return {
        "color": colormap(val),
        "weight": 3 if val > 0 else 1,
        "opacity": 0.9,
    }

folium.GeoJson(
    data=streets_wgs,
    name="Network (base_capacity)",
    style_function=style_function,
    tooltip=tooltip,
    highlight_function=lambda feat: {"weight": 5, "color": "#000000"}
).add_to(m)

# Add color scale and layer control
colormap.add_to(m)
folium.LayerControl().add_to(m)

# Save map to HTML file
m.save("network_gdf_base_capacity_map.html")


## 4.1. Map

For the proper presentation of interactive maps, the CRS of the used layers was changed to EPSG:4326 (WGS84).

In [ ]:
study_area_4326     = study_area_gdf.to_crs(4326)
exclusion_mask_4326 = exclusion_mask.to_crs(4326)
network_4326        = network_gdf.to_crs(4326)
street_lines_4326   = street_lines.to_crs(4326)
probes_4326         = probes.to_crs(4326)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# --- Study area outline ---
study_area_gdf.plot(
    ax=ax,
    edgecolor="#000000",
    facecolor="none",
    linewidth=1,
    zorder=3
)

# --- Street lines by status ---
# colors: original (blue), new (green), changed (greenish-blue/teal)
status_colors = {
    "original": "#000AC1",
    "changed": "#00C5C5",
    "new": "#009F03"
}

for status, color in status_colors.items():
    subset = street_lines[street_lines["_status"].str.lower() == status]
    if not subset.empty:
        subset.plot(
            ax=ax,
            edgecolor=color,
            linewidth=1,
            label=status.capitalize(),
            zorder=5
        )


# --- Basemap (kept underneath everything) ---
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=local_CRS, zorder=0)

# --- Legend ---
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color=status_colors["original"], lw=2, label="Original"),
    Line2D([0], [0], color=status_colors["changed"], lw=2, label="Changed"),
    Line2D([0], [0], color=status_colors["new"], lw=2, label="New"),
]

ax.legend(
    handles=legend_handles,
    loc="upper left",
    frameon=True,
    framealpha=0.9
)

ax.set_axis_off()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## 4.2. Summary statistics


In [ ]:
# Calculate descriptive statistics for the base street centerlines network

# Total number of segments
total_segments = len(street_lines)
# Number of segments by status
status_counts = street_lines["_status"].value_counts()
original_count = status_counts.get("original", 0)
changed_count = status_counts.get("changed", 0)
new_count = status_counts.get("new", 0)

# Percentage by status
original_pct = 100 * original_count / total_segments if total_segments else 0
changed_pct = 100 * changed_count / total_segments if total_segments else 0
new_pct = 100 * new_count / total_segments if total_segments else 0

# Calculate length (in meters) for each segment
if "length" not in street_lines.columns:
    street_lines["length"] = street_lines.geometry.length

# Total network length (km)
total_length_km = street_lines["length"].sum() / 1000
# Length by status (km)
original_length_km = street_lines.loc[street_lines["_status"] == "original", "length"].sum() / 1000
changed_length_km = street_lines.loc[street_lines["_status"] == "changed", "length"].sum() / 1000
new_length_km = street_lines.loc[street_lines["_status"] == "new", "length"].sum() / 1000

# Percentage of length by status
original_length_pct = 100 * original_length_km / total_length_km if total_length_km else 0
changed_length_pct = 100 * changed_length_km / total_length_km if total_length_km else 0
new_length_pct = 100 * new_length_km / total_length_km if total_length_km else 0

# Average segment length (m)
avg_length_m = street_lines["length"].mean()

# Display results

stats_df = pd.DataFrame({
    "Metric": [
        "Number of segments",
        "Original",
        "Changed",
        "New",
        "Network length (km)",
        "Original",
        "Changed",
        "New",
        "Average length of all segments (m)"
    ],
    "Value": [
        total_segments,
        original_count,
        changed_count,
        new_count,
        round(total_length_km, 2),
        round(original_length_km, 2),
        round(changed_length_km, 2),
        round(new_length_km, 2),
        round(avg_length_m, 2)
    ],
    "Percentage (%)": [
        100,
        round(original_pct, 2),
        round(changed_pct, 2),
        round(new_pct, 2),
        100,
        round(original_length_pct, 2),
        round(changed_length_pct, 2),
        round(new_length_pct, 2),
        ""
    ]
})

display(stats_df)

# 5. Street functions results


## 5.1. Results of the "link" function


### 5.1.1. Map


Below is an interactive map that visualizes the "link" function (edge betweenness centrality) of the street segments in the network. The street segments are color-coded based on their "link" function values, with a color gradient ranging from white (low "link" function) to dark red (high "link" function). The map also includes a legend to help interpret the color coding, and tooltips that display the exact "link" function value when hovering over a street segment.

In [ ]:
# Define your custom colors
custom_colors = ['#FFFFFF', "#FF0000", "#000000"]
cmap = LinearSegmentedColormap.from_list('custom_link', custom_colors)
norm = Normalize(vmin=0, vmax=1)

fig, ax = plt.subplots(figsize=(10, 10))

# --- Study area outline ---
study_area_gdf.plot(
    ax=ax,
    edgecolor="#000000",
    facecolor="none",
    linewidth=1,
    zorder=5
)

# Plot street segments colored by LINK value
street_lines.plot(
    ax=ax,
    linewidth=1.2,
    zorder=3,
    color=[to_hex(cmap(norm(v))) if not pd.isna(v) else "#cccccc" for v in street_lines["bet_centrality"]]
)

# --- Basemap ---
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=local_CRS, zorder=0)

# --- Colorbar ---
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.0435, pad=0.01)

ax.set_axis_off()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


### 5.1.2. Summary statistics


In [ ]:
plt.figure(figsize=(17,4))
plt.hist(street_lines.LINK, 
         bins=200, 
         range=(0, 1), 
         color="#ff0000",
         edgecolor="white",
         linewidth=0.5)
plt.margins(x=0, y=0)
plt.xticks(np.arange(0, 1.1, 0.1))
plt.show()

In [ ]:
link_descriptive = street_lines["LINK"].dropna()

# Define the stats
link_stats = {
    "Count": len(link_descriptive),
    "Mean": link_descriptive.mean(),
    "Mode": link_descriptive.mode().iloc[0],
    "Median (P50)": link_descriptive.median(),
    "Standard Deviation": link_descriptive.std(),
    "Q1 (P25)": link_descriptive.quantile(0.25),
    "Q3 (P75)": link_descriptive.quantile(0.75),
    "Interquartile Range (IQR)": link_descriptive.quantile(0.75) - link_descriptive.quantile(0.25),
    "P10": np.percentile(link_descriptive, 10),
    "P90": np.percentile(link_descriptive, 90),
    "P95": np.percentile(link_descriptive, 95),    
    "Skewness": skew(link_descriptive),
    "Kurtosis": kurtosis(link_descriptive)
}

# Put into a DataFrame for nice display
link_stats_df = pd.DataFrame(link_stats, index=["LINK"])
display(link_stats_df.T.round(4))

## 5.2. Results of the "place" function


### 5.2.1. Map


In [ ]:
# Define your custom colors
custom_colors = ['#FFFFFF', "#00FF00", "#000000"]
cmap = LinearSegmentedColormap.from_list('custom_PLACE', custom_colors)
norm = Normalize(vmin=0, vmax=1)

import geopandas as gpd
import folium

# --- Study area outline ---
study_area_gdf.plot(
    ax=ax,
    edgecolor="#000000",
    facecolor="none",
    linewidth=1,
    zorder=5
)

# Plot street segments colored by PLACE value
street_lines.plot(
    ax=ax,
    linewidth=1.2,
    zorder=3,
    color=[to_hex(cmap(norm(v))) if not pd.isna(v) else "#cccccc" for v in street_lines["PLACE"]]
)

# --- Basemap ---
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=local_CRS, zorder=0)

# --- Colorbar ---
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.0435, pad=0.01)

folium.LayerControl(collapsed=False).add_to(m)

m



### 5.2.2. Summary statistics


In [ ]:
plt.figure(figsize=(17,4))
plt.hist(street_lines.PLACE, 
         bins=200, 
         range=(0, 1), 
         color="#00ff00",
         edgecolor="white",
         linewidth=0.5)
plt.margins(x=0, y=0)
plt.xticks(np.arange(0, 1.1, 0.1))
plt.show()

In [ ]:
place_descriptive = street_lines["PLACE"].dropna()

# Define the stats
place_stats = {
    "Count": len(place_descriptive),
    "Mean": place_descriptive.mean(),
    "Mode": place_descriptive.mode().iloc[0],
    "Median (P50)": place_descriptive.median(),
    "Standard Deviation": place_descriptive.std(),
    "Q1 (P25)": place_descriptive.quantile(0.25),
    "Q3 (P75)": place_descriptive.quantile(0.75),
    "Interquartile Range (IQR)": place_descriptive.quantile(0.75) - place_descriptive.quantile(0.25),
    "P10": np.percentile(place_descriptive, 10),
    "P90": np.percentile(place_descriptive, 90),
    "P95": np.percentile(place_descriptive, 95),    
    "Skewness": skew(place_descriptive),
    "Kurtosis": kurtosis(place_descriptive)
}

# Put into a DataFrame for nice display
place_stats_df = pd.DataFrame(place_stats, index=["PLACE"])
display(place_stats_df.T.round(4))

# 6. Street typologies results


## 6.1. Map


In [ ]:
street_class_hex = {
    "I-E":   "#ff0000", "I-D":   "#ff4000", "I-C":   "#ff8000", "I-B":   "#ffbf00", "I-A":   "#ffff00",
    "II-E":  "#ff4040", "II-D":  "#ff8040", "II-C":  "#ffbf40", "II-B":  "#ffff40", "II-A":  "#bfff00",
    "III-E": "#ff8080", "III-D": "#ffbf80", "III-C": "#ffff80", "III-B": "#bfff40", "III-A": "#80ff00",
    "IV-E":  "#ffbfbf", "IV-D":  "#ffffbf", "IV-C":  "#bfff80", "IV-B":  "#80ff40", "IV-A":  "#40ff00",
    "V-E":   "#ffffff", "V-D":   "#bfffbf", "V-C":   "#80ff80", "V-B":   "#40ff40", "V-A":   "#00ff00"
}

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# --- Plot the study area outline ---
study_area_gdf.plot(
    ax=ax,
    edgecolor="black",
    facecolor="none",
    linewidth=1,
    zorder=5
)

# --- Plot streets ---
for lp_class, color in street_class_hex.items():
    subset = street_lines[street_lines["LP_class_abs"] == lp_class]
    if not subset.empty:
        subset.plot(ax=ax, linewidth=1.2, color=color, zorder=3)

# --- Basemap ---
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=local_CRS, zorder=0)

# --- Legend outside map frame, aligned top-right and tight to map ---
try:
    img = plt.imread("link_place_classes.png")

    # Adjust placement: closer to map (x0 smaller), higher (y0 larger)
    ax_legend = fig.add_axes([0.735, 0.635, 0.20, 0.20], zorder=30)
    # [x0, y0, width, height]
    ax_legend.imshow(img)
    ax_legend.axis("off")

except FileNotFoundError:
    ax.text(
        0.98, 0.02, "Legend image not found: link_place_classes.png",
        transform=ax.transAxes, va="bottom", ha="right",
        fontsize=9, color="red", zorder=30
    )

# --- Final layout tweaks ---
ax.set_axis_off()
ax.set_aspect('equal')

# Leave a bit of margin for legend on the right
plt.subplots_adjust(right=0.75, top=0.85, bottom=0.05, left=0.05)

plt.show()

In [ ]:
# Zoomed-in version of the LP-class map, centered on the street layer, without the side image.
# Reuses: study_area, street_lines, street_class_hex, ctx

fig, ax = plt.subplots(figsize=(10, 10))

# Fallback: compute 3763 layers if they don't already exist
try:
    _ = street_lines
except NameError:
    street_lines = street_lines_4326.to_crs(local_CRS)
try:
    _ = study_area
except NameError:
    study_area = study_area_4326.to_crs(local_CRS)

# --- Plot study area outline ---
study_area_gdf.plot(ax=ax, edgecolor="black", facecolor="none", linewidth=1, zorder=5)

# --- Plot streets by LP class ---
for lp_class, color in street_class_hex.items():
    subset = street_lines[street_lines["LP_class_abs"] == lp_class]
    if not subset.empty:
        subset.plot(ax=ax, linewidth=1.2, color=color, zorder=3)

# --- Basemap ---
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=local_CRS, zoom=15, zorder=0)

# --- Compute center and zoom in ---
minx, miny, maxx, maxy = street_lines.total_bounds
cx, cy = (minx + maxx) / 2.0, (miny + maxy) / 2.0

# Shift center east (right) and south (down)
offset_x = (maxx - minx) * 0.10   # move ~10% of map width east
offset_y = (maxy - miny) * 0.25   # move ~25% of map height south
cx = cx + offset_x
cy = cy - offset_y

# zoom_factor in (0,1]: smaller -> more zoomed in (0.5 shows half of the full extent)
zoom_factor = 0.35
full_width = max(maxx - minx, 1.0)   # avoid zero
full_height = max(maxy - miny, 1.0)

half_w = (full_width * zoom_factor) / 2.0
half_h = (full_height * zoom_factor) / 2.0

ax.set_xlim(cx - half_w, cx + half_w)
ax.set_ylim(cy - half_h, cy + half_h)

# --- Finalize ---
ax.set_axis_off()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# --- Plot the study area outline ---
study_area_gdf.plot(
    ax=ax,
    edgecolor="black",
    facecolor="none",
    linewidth=1,
    zorder=5
)

# --- Plot streets ---
for lp_class, color in street_class_hex.items():
    subset = street_lines[street_lines["LP_class_rel"] == lp_class]
    if not subset.empty:
        subset.plot(ax=ax, linewidth=1.2, color=color, zorder=3)

# --- Basemap ---
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=local_CRS, zorder=0)

# --- Legend outside map frame, aligned top-right and tight to map ---
try:
    img = plt.imread("link_place_classes.png")

    # Adjust placement: closer to map (x0 smaller), higher (y0 larger)
    ax_legend = fig.add_axes([0.735, 0.635, 0.20, 0.20], zorder=30)
    # [x0, y0, width, height]
    ax_legend.imshow(img)
    ax_legend.axis("off")

except FileNotFoundError:
    ax.text(
        0.98, 0.02, "Legend image not found: link_place_classes.png",
        transform=ax.transAxes, va="bottom", ha="right",
        fontsize=9, color="red", zorder=30
    )

# --- Final layout tweaks ---
ax.set_axis_off()
ax.set_aspect('equal')

# Leave a bit of margin for legend on the right
plt.subplots_adjust(right=0.75, top=0.85, bottom=0.05, left=0.05)

plt.show()

In [ ]:
# Zoomed-in version of the LP-class map, centered on the street layer, without the side image.
# Reuses: study_area, street_lines, street_class_hex, ctx

fig, ax = plt.subplots(figsize=(10, 10))

# Fallback: compute 3763 layers if they don't already exist
try:
    _ = street_lines
except NameError:
    street_lines = street_lines_4326.to_crs(local_CRS)
try:
    _ = study_area
except NameError:
    study_area = study_area_4326.to_crs(local_CRS)

# --- Plot study area outline ---
study_area_gdf.plot(ax=ax, edgecolor="black", facecolor="none", linewidth=1, zorder=5)

# --- Plot streets by LP class ---
for lp_class, color in street_class_hex.items():
    subset = street_lines[street_lines["LP_class_rel"] == lp_class]
    if not subset.empty:
        subset.plot(ax=ax, linewidth=1.2, color=color, zorder=3)

# --- Basemap ---
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=local_CRS, zoom=15, zorder=0)

# --- Compute center and zoom in ---
minx, miny, maxx, maxy = street_lines.total_bounds
cx, cy = (minx + maxx) / 2.0, (miny + maxy) / 2.0

# Shift center east (right) and south (down)
offset_x = (maxx - minx) * 0.10   # move ~10% of map width east
offset_y = (maxy - miny) * 0.25   # move ~25% of map height south
cx = cx + offset_x
cy = cy - offset_y

# zoom_factor in (0,1]: smaller -> more zoomed in (0.5 shows half of the full extent)
zoom_factor = 0.35
full_width = max(maxx - minx, 1.0)   # avoid zero
full_height = max(maxy - miny, 1.0)

half_w = (full_width * zoom_factor) / 2.0
half_h = (full_height * zoom_factor) / 2.0

ax.set_xlim(cx - half_w, cx + half_w)
ax.set_ylim(cy - half_h, cy + half_h)

# --- Finalize ---
ax.set_axis_off()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## 6.2. Summary statistics

In [ ]:
# Example data — replace with your own columns
x = street_lines["PLACE"]
y = street_lines["LINK"]

# Define figure and grid
fig = plt.figure(figsize=(8, 8))
gs = fig.add_gridspec(4, 4, wspace=0.05, hspace=0.05)

# === Main scatter plot ===
ax = fig.add_subplot(gs[1:4, 0:3])
# Map LP_class_rel to colors using street_class_hex
point_colors = street_lines["LP_class_rel"].map(street_class_hex).fillna("#cccccc")
ax.scatter(x, y, alpha=0.5, s=10, color=point_colors)
ax.set_xlabel("PLACE")
ax.set_ylabel("LINK")
ax.grid(True, linestyle='--', alpha=0.3)

# === Top histogram (x) ===
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax)
counts_x, bins_x, patches_x = ax_histx.hist(
    x,
    bins=100,
    color='#00ff00',
    alpha=0.7,
    edgecolor='white',
    linewidth=0.3
)
# Hide x-scale (shared with scatter), keep frequency scale
ax_histx.tick_params(axis='x', labelbottom=False, bottom=False)

# Make the tallest bin touch the ceiling: max bin == axis max
max_count_x = counts_x.max()
ax_histx.set_ylim(0, max_count_x)

# Show only half and max of the window (no 0)
ax_histx.set_yticks([max_count_x / 2, max_count_x])
ax_histx.tick_params(axis='y', direction='out', pad=4, labelsize=8)

# === Right histogram (y) ===
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax)
counts_y, bins_y, patches_y = ax_histy.hist(
    y,
    bins=100,
    orientation='horizontal',
    color='#ff0000',
    alpha=0.7,
    edgecolor='white',
    linewidth=0.3
)
# Hide y-scale (shared with scatter), keep frequency scale
ax_histy.tick_params(axis='y', labelleft=False, left=False)

# Make the tallest bin touch the ceiling: max bin == axis max
max_count_y = counts_y.max()
ax_histy.set_xlim(0, max_count_y)

# Show only half and max of the window (no 0)
ax_histy.set_xticks([max_count_y / 2, max_count_y])
ax_histy.tick_params(axis='x', direction='out', pad=4, labelsize=8)

# === Layout adjustments ===
ax.margins(x=0, y=0)
plt.tight_layout()
plt.show()


In [ ]:
# show all rows/columns and full cell width (use with care on very large outputs)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# then display the rows you want
street_lines.sort_values("LINK", ascending=False).head(100)